🧹 **Notebook Version: Clean (Output-Free)**

This is the **main executable version** of the EDA and scoring notebook. All output cells have been cleared to reduce file size and maintain GitHub compatibility.

- ✅ Run this notebook locally to generate all visualizations, charts, and Folium maps
- 📦 Uses `gdf_ca_cleaned.parquet` as the primary input
- 📤 Outputs will be saved to the `output/` and `maps/` folders

> 📘 For fully rendered outputs (maps, charts, scoring), see the preview notebooks below:
> - [`03b1_preview.ipynb`](03b1_preview.ipynb) – Dataset Overview & Thematic EDA  
> - [`03b2_preview.ipynb`](03b2_preview.ipynb) – Seismic Risk Analysis (Fault Proximity)  
> - [`03b3_preview.ipynb`](03b3_preview.ipynb) – Earthquake History & Composite Scoring
---

# 📊 Phase 3 – Exploratory Data Analysis & Underinsured Risk Score

**Notebook**: `03_eda_and_underinsured_score.ipynb`  
**Objective**: Explore the relationships between earthquake risk, insurance coverage, and demographic vulnerability across California counties. Derive a composite **underinsured risk score** to identify high-risk, underprotected regions.

---

### 📌 EDA Goals

- 📥 Load the cleaned dataset from the previous notebook  
- 📈 Visually and statistically explore relationships between key variables  
- 🕳️ Assess **coverage gaps** in high-risk earthquake zones  
- 🔍 Analyze interplay between:
  - 🧱 **Seismic hazard**  
    *(fault intersections, modeled loss, earthquake zone)*
  - 🛡️ **Insurance protection**  
    *(EQ take-up rate, homeowners insurance coverage, FEMA aid)*
  - 👥 **Socio-demographics**  
    *(poverty, housing age, seniors, household income, etc.)*
- 🧮 Derive a new composite variable: `underinsured_risk_score`

> This phase sets the foundation for **Phase 4: Predictive Modeling and Dashboard Visualizations**.

---

### 📤 Output Files

- `gdf_ca_scored_v4.geojson`: GeoJSON for mapping & visualization  
- `gdf_ca_scored_v4.parquet`: Full GeoDataFrame with types preserved (for modeling/EDA)  
📁 **Saved to**: `../output/`

---

### 🧾 Data Source

- ✅ **Input file**: `gdf_ca_cleaned.parquet`  
- 📁 **Filepath**: `../output/gdf_ca_cleaned.parquet`

---
## 📦 Load Packages


In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.patches as mpatches
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import seaborn as sns
import folium
from folium import Choropleth, CircleMarker, LayerControl
from branca.colormap import linear
from branca.element import Template, MacroElement
from scipy.stats import pearsonr, spearmanr
from sklearn.preprocessing import MinMaxScaler


---
## 📂 Data Loading


In [ ]:
# Load from Parquet (preserves full Python types)
gdf = gpd.read_parquet("../output/gdf_ca_cleaned.parquet")

# Ensure faults_list is a native Python list (not NumPy array)
gdf["faults_list"] = gdf["faults_list"].apply(
    lambda x: x.tolist() if isinstance(x, np.ndarray) else x
)

# Basic Overview
print("Dataset loaded successfully")
print("Shape:", gdf.shape)
print("CRS:", gdf.crs)

# Type check for validation
print("\nColumn Type Check — faults_list:")
print(gdf["faults_list"].apply(type).value_counts())

gdf.head()


In [ ]:
# Load seismic summary from Phase 1
eq_stats = pd.read_csv("../output/eq_stats_by_county.csv")

# Merge into main GeoDataFrame
gdf = gdf.merge(eq_stats, on="county", how="left")

# Confirm result
gdf[["county", "eq_count", "eq_mean_mag", "eq_max_mag"]].sort_values("eq_count", ascending=False).head(10)


---
## 📊 Part 1 – Dataset Overview & Initial Exploration

---
### 🔍 Dataset Overview 
Includes dataset structure, types, missing values, and merged seismic stats.

---
#### ➤ Dataset Structure & Types


In [ ]:
gdf.info()


In [ ]:
print("\nData Types Summary:")
print(gdf.dtypes.value_counts())

print("\nMissing Values (Top 10):")
print(gdf.isna().sum().sort_values(ascending=False).head(10))

print("\nColumn names:")
print(gdf.columns.tolist())

print("\nCounties in dataset:", gdf["county"].nunique())


In [ ]:
# Check Missing Azimuth & Fault Info
missing_azimuth = gdf[gdf["mean_fault_azimuth_deg"].isna()]
missing_azimuth[["county", "intersects_fault", "n_faults_intersecting", "faults_list"]]


In [ ]:
# Display counties with missing earthquake stats
missing_quake_data = gdf[gdf["eq_count"].isna()]
missing_quake_data[["county", "eq_count", "eq_max_mag", "eq_mean_mag"]]


In [ ]:
# Fill missing quake stats with zeros
gdf["eq_count"] = gdf["eq_count"].fillna(0).astype(int)
gdf["eq_max_mag"] = gdf["eq_max_mag"].fillna(0)
gdf["eq_mean_mag"] = gdf["eq_mean_mag"].fillna(0)

gdf[["eq_count", "eq_max_mag", "eq_mean_mag"]].isna().sum()


> ℹ️ **Note on Earthquake Stats:**  
> The following counties had **no recorded earthquakes** in the USGS dataset (1970–2020) and thus appear with zeros in `eq_count`, `eq_max_mag`, and `eq_mean_mag`:  
> • Calaveras  
> • Yuba  
> • Sutter  
> • Amador  
> These were originally `NaN` and filled with 0s to indicate absence of recorded seismic activity.


---
#### ➤ Summary Statistics


In [ ]:
# Transposed Summary Table
(gdf.describe().T
    .style.format({"count": "{:,.0f}", "mean": "{:,.2f}", "std": "{:,.2f}",
                    "min": "{:,.2f}", "25%": "{:,.2f}", "50%": "{:,.2f}",
                    "75%": "{:,.2f}", "max": "{:,.2f}"})
    .set_caption("Summary Statistics (Formatted)")
)


---
#### ➤ Selected Histograms


In [ ]:
# Quick Histograms of Key Variables
gdf[["eq_takeup_pct", "built_pre_1980", "eq_count", "american_indian_alone_pct"]].hist(bins=20, figsize=(12, 4), edgecolor='black')

plt.tight_layout()
plt.savefig("../output/plot_hist_skewed_vars_quickview.png", dpi=300)


---
#### ➤ Fault Exposure by County


In [ ]:
# Bar Chart of Fault Counts
gdf.sort_values("n_faults_intersecting", ascending=False).plot.bar(
    x="county", y="n_faults_intersecting", color="tomato", edgecolor="black",
    figsize=(12, 6), title="Number of Fault Zones Intersecting Each County")
plt.ylabel("Count of Faults")
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig("../output/plot_bar_fault_counts.png", dpi=300)
plt.show()


---
#### ➤ Seismic Risk Comparison: Fault Lines vs Earthquake Activity


In [ ]:
# Set up figure and axes
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- Map 1: Fault Line Exposure ---
gdf.plot(
    column="n_faults_intersecting",
    cmap="OrRd",
    legend=True,
    edgecolor="black",
    ax=axes[0]
)
axes[0].set_title("Fault Line Exposure\n(Number of Fault Zones per County)")
axes[0].axis("off")

# --- Map 2: Earthquake Frequency ---
gdf.plot(
    column="eq_count",
    cmap="viridis",
    legend=True,
    edgecolor="black",
    ax=axes[1]
)
axes[1].set_title("Earthquake Frequency\n(Total Quakes 1970–2020 per County)")
axes[1].axis("off")

plt.suptitle("Seismic Risk Comparison: Fault Lines vs Earthquake Activity", fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig("../output/plot_fault_lines_vs_quake.png", dpi=300)
plt.show()


---
### 📈 Skewed Distributions – Identify High-Variance Variables  
- Function: `identify_high_spread_vars()`  
- Top 15 high-spread numeric variables

---
#### ➤ Identify High Spread Variables


In [ ]:
# Preview variables with large spread
def identify_high_spread_vars(df, top_n=15):
    numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns
    stats = df[numeric_cols].describe().T
    stats["spread_ratio"] = stats["max"] / stats["mean"]
    stats = stats.replace([np.inf, -np.inf], np.nan).dropna(subset=["spread_ratio"])
    return stats.sort_values("spread_ratio", ascending=False).round(2).head(top_n)

high_spread = identify_high_spread_vars(gdf, top_n=15)
high_spread[["count", "mean", "std", "max", "spread_ratio"]]


---
### 🏛️ Investigation of Skewed Variable — `fema_funding_earthquake`

In this section, I analyzed the amount of FEMA earthquake-related public assistance funding allocated to each California county. While not a direct predictor of earthquake insurance take-up, this variable offers insight into **where major seismic disasters have occurred** and how federal aid has been distributed over time.

---
#### ➤ Distribution of FEMA Earthquake Funding


In [ ]:
sns.histplot(gdf["fema_funding_earthquake"], bins=30, kde=True)
plt.title("Distribution of FEMA Earthquake Funding")
plt.xlabel("FEMA Funding ($)")
plt.show()


In [ ]:
sns.histplot(np.log1p(gdf["fema_funding_earthquake"]), bins=30, kde=True)
plt.title("Log-Transformed FEMA Funding")
plt.xlabel("log(FEMA Funding + 1)")
plt.show()


> The distribution of `fema_funding_earthquake` is **highly right-skewed**.  
> - Most counties received **little or no FEMA funding** over the period of record  
> - A small number of counties received **very large amounts**, often following major seismic events  
> - The maximum recorded total exceeded **\$44 million**
>
> After applying a log transformation, the distribution becomes more normalized — but still reflects the **event-driven nature** of this variable
>
> This strongly suggests that FEMA funding is **reactive** (post-disaster), rather than tied to long-term seismic risk or baseline vulnerability.  
> As a result, I use this variable for **narrative exploration**, but **exclude it from modeling.**

---
#### ➤ FEMA Funding vs. Fault Line Proximity


In [ ]:
# Are counties with high FEMA funding close to fault lines?
gdf[gdf["fema_funding_earthquake"] > 0][["county", "fema_funding_earthquake", "nearest_fault_km"]] \
    .sort_values(by="fema_funding_earthquake", ascending=False) \
    .head(10)  


In [ ]:
sns.scatterplot(
    data=gdf, 
    x="centroid_to_fault_km", 
    y="fema_funding_earthquake"
)
plt.title("FEMA Funding vs Distance to Nearest Fault")
plt.show()


> Counties that received the **largest FEMA earthquake allocations** are located **directly on or adjacent to major fault zones** — often with `nearest_fault_km = 0.00`.
>
> Examples include:
> - **San Luis Obispo**: \$44M
> - **Imperial**: \$25M
> - **Napa County**: \$21M (likely related to the 2014 South Napa Earthquake)
>
> These cases reinforce the idea that FEMA assistance is closely tied to **localized damage** caused by major seismic events — not by fault proximity alone.
>
> **Insight**: Proximity to faults is necessary but not sufficient for FEMA support — funding is conditional on **disaster declarations** following specific events.

---
#### ➤ FEMA Funding vs. Earthquake Frequency


In [ ]:
# Earthquake frequency + FEMA funding preview
gdf[["county", "eq_count", "fema_funding_earthquake"]].sort_values(by="eq_count", ascending=False).head(10)


>**Key Observation**  
>Many of the counties with the **most frequent earthquakes** — including **San Bernardino (509)**, **Mono (374)**, and **Kern (354)** — received **no FEMA earthquake funding** at all.  
>In contrast, **Imperial County**, which ranked 4th in frequency (317 quakes), received over **\$25 million** in FEMA support. This suggests that:
>- FEMA does **not respond to frequency alone**  
>- **Magnitude**, **infrastructure damage**, and **population vulnerability** are critical in triggering federal assistance  
>- Counties with **frequent but low-impact quakes** (e.g., Mono, San Bernardino) may **not meet FEMA’s disaster declaration thresholds**  
>
>For example:
>- **Imperial** received funding likely due to structural impacts in population centers
>- **Napa County**, with fewer quakes overall, received **$21M** following a single high-impact event in 2014
>
>Insight: FEMA funding is more closely aligned with **disaster severity** than seismic frequency — reinforcing its role as a **reactive aid mechanism**, not a proxy for ongoing risk.


---
#### ➤ FEMA Earthquake Funding Map


In [ ]:
# FEMA Earthquake Funding Map
gdf.plot(
    column="fema_funding_earthquake", 
    cmap="OrRd", 
    legend=True, 
    edgecolor="black",
    figsize=(8, 6)
)
plt.title("FEMA Earthquake Funding by County")
plt.axis("off")
plt.savefig("../output/plot_fema.png", dpi=300)
plt.show()


> The funding map clearly illustrates a **clustered pattern**: only a handful of counties have received significant FEMA earthquake funds, with most of the state showing no historical allocation. These spatial disparities highlight the **event- and impact-specific nature** of FEMA’s role, and reinforce the decision to treat this variable as **contextual only**, not as a predictive feature.

---

#### ➤ ✅ Conclusion
- `fema_funding_earthquake` reflects **past disaster response**, not long-term exposure
- It is **not a reliable or generalizable input** for modeling earthquake insurance take-up
- I include this variable for **narrative storytelling and map visualization**, but **exclude it from the core predictive model**


---
### 📍 Investigation of Skewed Variable — `nearest_fault_km`

In this section, I evaluated two proximity-based variables related to fault lines:

- `nearest_fault_km`: minimum distance from any part of the county to the nearest fault line (may be 0 km if intersecting)
- `centroid_to_fault_km`: distance from the county centroid to the nearest fault line

My goal was to determine which of these variables better captures county-level exposure to seismic risk and helps explain variation in earthquake insurance take-up rates.

---
#### ➤ Distance to Nearest Fault Line (`nearest_fault_km`)


In [ ]:
# Top counties farthest from fault lines (based on nearest_fault_km)
display(gdf[["county", "nearest_fault_km"]].sort_values("nearest_fault_km", ascending=False).head(10))


>I examined the `nearest_fault_km` variable to identify counties geographically farthest from fault lines:
>- **Del Norte County** has the maximum value: **27.21 km**
>- Many counties (e.g., **Butte, Riverside, Kern**) report **0.00 km**, indicating direct intersection
>- Urban counties like **Sacramento** show moderate distance (~7.3 km)
>- Counties such as **Mariposa** and **Sutter** fall into the 10–15 km range  
>  
>However, this variable exhibits **severe right skew**, with most counties tightly clustered near 0 km.

---

>#### ❌ Why I Didn't Use `nearest_fault_km`
>
> Despite initial exploration, I excluded `nearest_fault_km` from further analysis and modeling due to several limitations:
>- **Low variability** — Majority of counties report a value of **0.0 km**
>- **Redundancy** — Overlaps conceptually with `intersects_fault` (a boolean indicator already included)
>- **Weak explanatory power** — Does not effectively differentiate counties based on perceived seismic exposure
>
>Instead, I used `centroid_to_fault_km`, which provides a **continuous**, **non-zero**, and more **informative** measure of proximity to fault lines.

---
#### ➤ EQ Take-Up Rate vs. Centroid-to-Fault Distance


In [ ]:
# Counties with farthest centroid-based distance
display(gdf[["county", "centroid_to_fault_km"]].sort_values("centroid_to_fault_km", ascending=False).head(10))


In [ ]:
# Scatterplot: EQ Take-Up Rate vs. Distance to Fault
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=gdf, 
    x="centroid_to_fault_km", 
    y="eq_takeup_pct"
)
plt.title("EQ Take-Up Rate vs. Distance to Nearest Fault (Centroid-Based)")
plt.xlabel("Distance to Nearest Fault (km)")
plt.ylabel("Earthquake Take-Up Rate (%)")
plt.tight_layout()
plt.show()


> Based on the scatterplot, there is no clear linear relationship between `centroid_to_fault_km` and earthquake insurance take-up rates.  
> Counties located very close to fault lines (0–5 km) show a wide range of take-up rates, from under 2% to over 50%.  
> While I expected to see higher take-up rates in areas closer to faults, the scatter remains dispersed even at short distances.  
> This suggests that **proximity to fault lines alone does not strongly predict insurance behavior**, and other factors (e.g., income, awareness, housing type) may be more influential.


In [ ]:
# Regression Scatterplot
plt.figure(figsize=(8, 5))
sns.regplot(
    data=gdf, 
    x="centroid_to_fault_km", 
    y="eq_takeup_pct",
    scatter_kws={'alpha': 0.7},
    line_kws={'color': 'red'}
)
plt.title("EQ Take-Up Rate vs. Distance to Nearest Fault (Centroid-Based)")
plt.xlabel("Distance to Nearest Fault (km)")
plt.ylabel("Earthquake Take-Up Rate (%)")
plt.tight_layout()
plt.show()

# Calculate correlation
pearson_corr = gdf["centroid_to_fault_km"].corr(gdf["eq_takeup_pct"], method="pearson")
spearman_corr = gdf["centroid_to_fault_km"].corr(gdf["eq_takeup_pct"], method="spearman")

print(f"📉 Pearson Correlation: {pearson_corr:.2f}")
print(f"📉 Spearman Correlation: {spearman_corr:.2f}")


> I added a regression line to assess whether there’s a trend between `centroid_to_fault_km` and earthquake insurance take-up rates.  
> Visually, the slope is slightly negative, but the scatter remains broad across all distance ranges.  
>
> To confirm this, I calculated both Pearson and Spearman correlations:
> - **Pearson**: -0.34
> - **Spearman**: -0.64
>
> These values indicate a **weak negative relationship** — suggesting that counties located farther from fault lines tend to have slightly lower take-up rates, but the association is not strong.  
> Overall, **fault proximity alone does not appear to be a reliable predictor** of take-up behavior and likely interacts with other demographic or socioeconomic variables.


In [ ]:
# Binned barplot of take-up rates by centroid distance
gdf["distance_bin"] = pd.cut(
    gdf["centroid_to_fault_km"],
    bins=[0, 5, 10, 20, 40, 80],
    labels=["0–5 km", "5–10 km", "10–20 km", "20–40 km", "40–80 km"])

plt.figure(figsize=(8, 5))
sns.barplot(
    data=gdf, 
    x="distance_bin", 
    y="eq_takeup_pct", 
    hue="distance_bin",
    palette="viridis", 
    legend=False
)
plt.title("Avg EQ Take-Up Rate by Distance to Fault Line")
plt.xlabel("Centroid-to-Fault Distance Bin")
plt.ylabel("EQ Take-Up Rate (%)")
plt.tight_layout()
plt.show()


>**Key Insight** — High Variance in 10–20 km Bin
>
>- This distance group shows the **widest error bars** in the barplot
>- Most counties in this bin range between **4–13%** take-up rate
>- But extreme values in counties like:
>   - **Mono County** → 52.3%
>   - **San Diego County** → 23.2%
>- Others like **Imperial** and **Glenn** fall below 2.5%
>
>These **outliers inflate the mean (~12%)** and highlight the importance of **examining distribution**, not just averages.

---

In [ ]:
# View counties in the 10–20 km distance bin
gdf[gdf['distance_bin'] == '10–20 km'][['county', 'eq_takeup_pct']]


---
#### ➤ ✅ Conclusion

- `nearest_fault_km` was excluded due to severe skew and limited usefulness
- `centroid_to_fault_km` provides more **granular**, **meaningful**, and **spatially distributed** insight into fault proximity
- This variable was retained for modeling and risk scoring
  

---
### 🏚️ Investigation of Skewed Variable — `built_pre_1980`

In this section, I examined the number and percentage of housing units built before 1980, a key proxy for structural vulnerability. Older homes are more likely to lack seismic retrofitting, making them more sensitive to earthquake damage — and potentially influencing insurance demand.

I focused on:
- Raw distribution and top counties
- Derived variable: `pre_1980_pct` (percent of total housing)
- Correlation with earthquake insurance take-up
- Outlier analysis (Mono County)
- County-level comparison chart
- ✅ *Conclusion: Moderate correlation, stronger without Mono*

---

#### ➤ Distribution of Pre-1980 Housing Units


In [ ]:
# Histogram: Distribution of pre-1980 housing units
plt.figure(figsize=(8, 5))
sns.histplot(gdf['built_pre_1980'], bins=20, kde=True, color='teal')
plt.title('Distribution of Pre-1980 Housing Units (All Counties)')
plt.xlabel('Housing Units Built Before 1980')
plt.ylabel('Number of Counties')
plt.grid(True)
plt.tight_layout()
plt.show()


> The distribution of `built_pre_1980` is highly right-skewed, with most counties falling below 100,000 older units.  
> A small number of large-population counties dominate the upper tail — particularly Los Angeles.

---
#### ➤ Top Counties by Pre-1980 Units


In [ ]:
# Top 20 Counties with Most Pre-1980 Units (in thousands)
top_n = 20
gdf_sorted = gdf.sort_values("built_pre_1980", ascending=False).head(top_n)

plt.figure(figsize=(12, 6))
ax = sns.barplot(
    data=gdf_sorted,
    x="county",
    hue="county",
    y=gdf_sorted["built_pre_1980"] / 1000,
    palette="viridis"
)
ax.set_title("Top 20 Counties by Pre-1980 Housing Units", fontsize=14)
ax.set_ylabel("Units (Thousands)")
ax.set_xlabel("County")
plt.xticks(rotation=45, ha='right')
ax.bar_label(ax.containers[0], fmt="%.0f", padding=3)
for container in ax.containers:
    ax.bar_label(container, fmt="%.0f", padding=3, label_type="edge")
plt.tight_layout()
plt.show()


> **Los Angeles County** is a major outlier, with over **2.5 million** pre-1980 housing units — nearly **4x** more than any other county.  
> This dominant stock of legacy buildings may affect both seismic vulnerability and insurance behavior, especially if retrofitting policies are unevenly enforced or adopted.

---
#### ➤ Derived Variable: `pre_1980_pct`


In [ ]:
# Add % of Pre-1980 Housing
gdf["pre_1980_pct"] = (gdf["built_pre_1980"] / gdf["total_housing_units"]) * 100
gdf["pre_1980_pct"] = gdf["pre_1980_pct"].round(2)
gdf[["county","pre_1980_pct"]].sort_values("pre_1980_pct", ascending=False).head(10)


> I calculated the share of older housing (`pre_1980_pct`) by dividing pre-1980 units by total housing stock in each county.  
> This normalized metric removes size bias and provides a better basis for comparing counties of different sizes.

---
#### ➤ Correlation with EQ Take-Up Rate


In [ ]:
# Scatterplot: Pre-1980 Housing % vs EQ Take-Up %
plt.figure(figsize=(10, 6))
sns.regplot(
    data=gdf,
    x="pre_1980_pct",
    y="eq_takeup_pct",
    scatter_kws={"s": 60, "alpha": 0.8},
    line_kws={"color": "red"},
)
plt.title("Pre-1980 Housing (%) vs Earthquake Insurance Take-Up (%)")
plt.xlabel("Pre-1980 Housing (% of Total Units)")
plt.ylabel("EQ Take-Up Rate (%)")
plt.grid(True)
plt.tight_layout()
plt.show()


> The scatterplot suggests a **positive relationship** between `pre_1980_pct` and earthquake insurance take-up:  
> - Counties with a **higher share of older homes** generally report **higher insurance take-up rates**
> - However, the pattern is not tightly linear — there is visible dispersion across the middle of the distribution

---
#### ➤ Outlier Analysis: Mono County


In [ ]:
# Correlation Summary (with/without Mono County)
pearson_full = gdf["pre_1980_pct"].corr(gdf["eq_takeup_pct"], method="pearson")
spearman_full = gdf["pre_1980_pct"].corr(gdf["eq_takeup_pct"], method="spearman")

gdf_no_mono = gdf[gdf["county"] != "Mono"]
pearson_no_mono = gdf_no_mono["pre_1980_pct"].corr(gdf_no_mono["eq_takeup_pct"], method="pearson")
spearman_no_mono = gdf_no_mono["pre_1980_pct"].corr(gdf_no_mono["eq_takeup_pct"], method="spearman")

print(f"Pearson (with Mono): {pearson_full:.2f}")
print(f"Pearson (no Mono) : {pearson_no_mono:.2f}")
print(f"Spearman (with Mono): {spearman_full:.2f}")
print(f"Spearman (no Mono) : {spearman_no_mono:.2f}")


> **Mono County** is a clear outlier, with:
> - **52.3%** earthquake insurance take-up (highest in the state)
> - **51.0%** of housing units built before 1980  
>
> To evaluate its impact, I computed Pearson and Spearman correlations with and without Mono County:
>
> - **Pearson correlation** increased from **0.22 → 0.32** (Mono lowers the linear strength)
> - **Spearman correlation** stayed constant at **0.36**
>
> This suggests that **Mono inflates variance**, weakening the linear association — although the rank-based (monotonic) trend holds.  
> I may consider flagging or down-weighting such outliers in future modeling.

---
#### ➤ County Comparison: Pre-1980 Housing vs EQ Take-Up


In [ ]:
# Side-by-Side Barplot: Pre-1980 Housing vs EQ Take-Up
df_plot = gdf[["county", "pre_1980_pct", "eq_takeup_pct"]].copy()
df_plot = df_plot.sort_values("pre_1980_pct", ascending=False)
df_melted = df_plot.melt(id_vars="county", var_name="Variable", value_name="Value")

plt.figure(figsize=(12, 10))
sns.barplot(
    data=df_melted,
    x="Value",
    y="county",
    hue="Variable",
    palette=["#2C7BB6", "#FDAE61"]
)
plt.title("Pre-1980 Housing vs EQ Insurance Take-Up Rate by County", fontsize=14)
plt.xlabel("Percentage (%)")
plt.ylabel("County")
plt.legend(title="Variable", loc="lower right")
plt.grid(axis="x", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()


> The grouped bar chart shows that many counties with **higher shares of older housing** also exhibit **higher take-up rates**.  
> - **Mono** and **San Luis Obispo** both rank high on both metrics  
> - Other counties (e.g., **Sierra**, **Alpine**) have high pre-1980 housing shares but low take-up — suggesting **other barriers**, like income or awareness, may suppress insurance uptake.
>
> Overall, this variable appears to be **moderately correlated** with take-up behavior and may help identify structurally vulnerable, underinsured regions.


In [ ]:
# Preview Top 10 by EQ Take-Up
gdf[["county", "pre_1980_pct", "eq_takeup_pct"]].sort_values("eq_takeup_pct", ascending=False).head(10)


---
#### ➤ ✅ Conclusion  
- `pre_1980_pct` offers more analytical value than the raw count  
- Moderate correlation with insurance take-up — especially when outliers like Mono are excluded  
- A useful variable for modeling underinsurance and seismic vulnerability


---

### 🧬 Investigation of Skewed Variable — `american_indian_alone_pct`

In this section, I explored the relationship between the percentage of residents identifying as American Indian (alone) and earthquake insurance take-up rates. While this population represents a small share in most counties, certain rural areas in Northern and Eastern California have significantly higher proportions.

---

### 🧑‍🦲 American Indian Population (%)  
- Scatterplot with annotations  
- Binned analysis and grouped means  
- Pearson & Spearman correlation  
- ✅ *Conclusion: Weak-to-moderate pattern; high variance*

---
#### ➤ Visual Patterns and Key Counties


In [ ]:
# Scatter Plot: American Indian % vs EQ Take-Up %
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=gdf,
    x="american_indian_alone_pct",
    y="eq_takeup_pct",
    s=80,
    color="purple",
    edgecolor="black"
)

# Annotate counties with high American Indian population
high_native = gdf[gdf["american_indian_alone_pct"] > 5]
for _, row in high_native.iterrows():
    plt.text(
        row["american_indian_alone_pct"] + 0.1,
        row["eq_takeup_pct"],
        row["county"],
        fontsize=9
    )

# Manually highlight notable counties
highlight = ["Mono", "Alpine", "Inyo", "Del Norte"]
for _, row in gdf[gdf["county"].isin(highlight)].iterrows():
    plt.text(
        row["american_indian_alone_pct"] + 0.1,
        row["eq_takeup_pct"],
        row["county"],
        fontsize=9,
        color="black"
    )

# Formatting
plt.xlabel("American Indian Population (%)")
plt.ylabel("Earthquake Insurance Take-Up Rate (%)")
plt.title("EQ Take-Up vs American Indian Population")
plt.grid(True)
plt.tight_layout()
plt.show()


> The scatterplot revealed that:
> - Most counties have **less than 3%** American Indian population
> - A few counties stand out, including:
>   - **Alpine** (24.6%)
>   - **Inyo** (8.1%)
>   - **Del Norte** (6.2%)
>
> These counties show **moderate to high take-up rates**, but overall, there’s **no strong visual trend** across the full dataset.
> 
> **Mono County** remains an outlier with **52.3% take-up** but only **1.2% American Indian population**, indicating that other factors are likely driving its behavior.


---
#### ➤ Top Counties by American Indian Population Share


In [ ]:
# Sort and display counties with highest American Indian %
gdf[["county", "american_indian_alone_pct"]].sort_values(by="american_indian_alone_pct", ascending=False).head()


> Counties with the highest percentages of American Indian residents tend to be **rural and geographically isolated**, such as Alpine, Inyo, and Del Norte.  
> These areas may have distinct housing, risk perception, or tribal insurance considerations not captured in statewide models.

---
#### ➤ Correlation Analysis


In [ ]:
# Pearson (linear) and Spearman (rank) correlation
pearson_corr, _ = pearsonr(gdf["american_indian_alone_pct"], gdf["eq_takeup_pct"])
spearman_corr, _ = spearmanr(gdf["american_indian_alone_pct"], gdf["eq_takeup_pct"])

print(f"Pearson correlation: {pearson_corr:.2f}")
print(f"Spearman correlation: {spearman_corr:.2f}")


> To quantify the relationship:
> - **Pearson correlation** (linear): **0.20**
> - **Spearman correlation** (rank-based): **-0.31**
>
> This suggests a **conflicting picture**:
> - The linear trend is slightly positive, driven by a few counties with both higher Native population and higher take-up
> - However, the **negative Spearman correlation** implies that higher Native population **does not consistently correspond** with higher take-up when ranked — likely due to wide variability

---
#### ➤ Binned Analysis: Take-Up by American Indian %


In [ ]:
# Bin counties by American Indian %
bins = [0, 1, 3, 10, 25]
labels = ["<1%", "1–3%", "3–10%", "10–25%"]
gdf["native_pct_bin"] = pd.cut(gdf["american_indian_alone_pct"], bins=bins, labels=labels)

# Compute grouped summary statistics
summary = (
    gdf.groupby("native_pct_bin", observed=False)["eq_takeup_pct"]
    .agg(["count", "mean", "std"])
    .reset_index()
    .rename(columns={
        "native_pct_bin": "American Indian % Bin",
        "count": "Counties",
        "mean": "Avg Take-Up %",
        "std": "Std Dev"
    })
)

# Display summary as styled table
summary.style.format({"Avg Take-Up %": "{:.1f}", "Std Dev": "{:.1f}"})


In [ ]:
# Barplot with error bars for standard deviation
plt.figure(figsize=(10, 6))
ax = sns.barplot(
    data=summary,
    x="American Indian % Bin",
    y="Avg Take-Up %",
    yerr=summary["Std Dev"],
    color="#88c",
    edgecolor="black"
)

# Annotate bars with exact mean values
for i, row in summary.iterrows():
    ax.text(i, row["Avg Take-Up %"] + 0.5, f"{row['Avg Take-Up %']:.1f}%", ha="center", fontsize=10)

# Formatting
plt.title("Avg EQ Take-Up Rate by % American Indian Population", fontsize=14)
plt.xlabel("% American Indian Population (Binned)")
plt.ylabel("Avg Earthquake Insurance Take-Up Rate (%)")
plt.tight_layout()
plt.show()


> I grouped counties into bins based on their American Indian population percentage:
> - **<1%**
> - **1–3%**
> - **3–10%**
> - **10–25%**
>
> The grouped bar chart showed a **general upward trend** — bins with higher Native American percentages tended to have higher average take-up rates.
>
> However, **standard deviations were large** in each group, which suggests:
> - There’s **high within-group variability**
> - The observed trend is **not consistent** and may be driven by a small number of influential counties


---
#### ➤ ✅ Conclusion
- The variable `american_indian_alone_pct` shows **some positive association** with EQ insurance take-up, but the relationship is **weak and noisy**
- Results are **sensitive to outliers**, and the overall trend is not consistent across bins or ranks
- I may consider **interacting this variable with others** (e.g., rural status, housing age) in future models to better understand its impact

---
### ✅ Summary – Part 1: Dataset Overview & Initial Exploration

In the first phase of my exploratory analysis, I focused on understanding the **spread and skewness** of key variables in the dataset — particularly those related to housing stock, demographics, seismic risk, and disaster history.

Several variables showed **extremely high variance**, especially raw count variables such as `fema_funding_earthquake`, `built_pre_1980`, and `total_population`.  
This is expected due to the **large differences in county size** — with urban areas like Los Angeles dwarfing rural counties like Alpine or Sierra.

To address this imbalance and enable **more meaningful comparisons across counties**, I derived and shifted focus toward **percentage-based or normalized variables**, including:

- `pre_1980_pct` – % of housing built before 1980  
- `american_indian_alone_pct` – % identifying as American Indian (alone)  
- `below_poverty_pct` – % of population living below poverty line  
- `eq_takeup_pct` – % of residential units with earthquake insurance  

These variables offer a **relative, size-independent view** of housing vulnerability, demographic sensitivity, and insurance behavior.

This foundation will support deeper, theme-specific exploration in **Part 2**, where I group variables by topic (e.g., demographics, housing, risk exposure) and analyze their relationships with earthquake insurance take-up rates.


---
## 📊 Part 2 – Thematic EDA by Variable Group

___

### 📊 Percentage-Based Variables

To ensure fair comparisons across counties of different population sizes, we prioritize percentage-based variables in our exploratory analysis. These include:

- **Demographics:** Age, Race, Ethnicity
- **Housing Tenure:** Owner/Renter distribution, Pre-1980 housing
- **Insurance & Poverty:** Earthquake take-up rate, Loss ratios, Homeowners coverage, Poverty levels
- **Vacancy Characteristics:** Various categories of housing vacancy


In [ ]:
# Find all columns that include 'pct' in their name
pct_columns = [col for col in gdf.columns if 'pct' in col.lower()]

# Display the list
print("Percentage-based variables in the dataset:")
for col in pct_columns:
    print(f"• {col}")

---
### Group 1: Age Distribution

In this section, I analyzed the age composition of each California county and its potential relationship to earthquake insurance behavior.

I focused on three main variables:
- `age_0_17_pct` – share of the population under 18
- `age_65_up_pct` – share of the population aged 65 and older
- `senior_minus_youth` – the difference between seniors and youth


In [ ]:
# Select age variables
age_cols = ["age_0_17_pct", "age_18_64_pct", "age_65_up_pct"]
gdf[age_cols].describe().T.style.format("{:,.2f}")


> **Quick Observations – Age Composition**
> - Most counties are **majority working-age (18–64)**, with an average of **~59%**
> - The **child population** (`age_0_17_pct`) ranges from **~13% to 30%**
> - The **senior population** (`age_65_up_pct`) has a wider spread — from **~10% to 34.5%**
>
> While the range is meaningful, age group percentages show **relatively moderate spread** compared to high-variance variables like population or FEMA funding. These normalized values are **well-suited for cross-county comparison**, especially in the context of behavioral modeling.

---
#### ➤ Age Group Distribution by County (Stacked View)
Stacked barplot showing the composition of age groups across California counties.


In [ ]:
age_df = gdf[["county"] + age_cols].copy()
age_long = age_df.melt(id_vars="county", var_name="age_group", value_name="percentage")

plt.figure(figsize=(14, 6))
sns.barplot(data=age_long, x="county", y="percentage", hue="age_group")
plt.title("Age Group Distribution by County (%)")
plt.ylabel("Percentage")
plt.xlabel("County")
plt.xticks(rotation=90)
plt.legend(title="Age Group")
plt.tight_layout()
plt.show()


> **Key Takeaways from the Age Distribution Plot**
> - **Majority Age Group**:  
>   In every county, the largest age group is **18–64 years**, representing the working-age population. Most counties fall between **55–65%** in this category.
> - **Youth vs. Seniors**:  
>   In most counties, the share of **children (0–17)** exceeds that of **seniors (65+)**.  
>   However, a few rural or retirement-oriented counties — such as **Sierra**, **Mariposa**, and **Amador** — have a **higher percentage of seniors than youth**.
> - **Implications for Earthquake Insurance**:  
>   Older populations may be **more risk-aware**, potentially leading to **higher insurance interest**, but they may also face **affordability barriers**, especially in fixed-income households.  
>   Counties with more youth may reflect **family-oriented populations** or **new homeowners**, which could influence insurance adoption differently.


---
#### ➤ Correlation with EQ Take-Up Rate
Pearson and Spearman correlations between age group percentages and earthquake insurance take-up rate.


In [ ]:
# Select variables
age_vars = ["age_0_17_pct", "age_18_64_pct", "age_65_up_pct"]

# Compute correlations
for var in age_vars:
    pearson_corr, _ = pearsonr(gdf[var], gdf["eq_takeup_pct"])
    spearman_corr, _ = spearmanr(gdf[var], gdf["eq_takeup_pct"])
    print(f"📊 {var}")
    print(f"   • Pearson  : {pearson_corr:.2f}")
    print(f"   • Spearman : {spearman_corr:.2f}\n")


> **Age Demographics and Earthquake Insurance Take-Up**
> - **Youth (0–17 years)**:  
>   Shows a **moderate negative correlation** with take-up. This may reflect **lower perceived risk** or **budget limitations** among families with children.
> - **Working-Age Adults (18–64 years)**:  
>   Displays the **strongest positive correlation** with take-up. Likely related to **homeownership**, **income stability**, and **asset protection behavior**.
> - **Seniors (65+ years)**:  
>   Shows **very weak or no linear correlation**. Suggests mixed take-up behavior — possibly influenced by fixed income, home equity, or regional factors.
>
> These results support the idea that **insurance awareness and affordability interventions** may be especially important in **younger, family-oriented counties**, where take-up rates are currently lower.

---
#### ➤ Identify Counties Where Seniors Outnumber Youth
Flag counties where the senior population percentage exceeds that of youth, indicating potential age-skewed demographics.


In [ ]:
# Filter counties where seniors outnumber youth
gdf["senior_minus_youth"] = gdf["age_65_up_pct"] - gdf["age_0_17_pct"]
gdf["seniors_gt_youth"] = gdf["senior_minus_youth"] > 0

senior_dominant = gdf[gdf["seniors_gt_youth"]].copy()
senior_dominant = senior_dominant.sort_values("eq_takeup_pct", ascending=False)

# Preview
print("Counties where seniors (65+) outnumber youth (0–17):")
display(senior_dominant[["county", "age_65_up_pct", "age_0_17_pct", "senior_minus_youth", "eq_takeup_pct"]])


In [ ]:
# Barplot to compare youth vs. senior % in those counties
plt.figure(figsize=(10, 6))
df_melt = senior_dominant.melt(
    id_vars="county",
    value_vars=["age_0_17_pct", "age_65_up_pct"],
    var_name="age_group",
    value_name="percentage"
)

sns.barplot(
    data=df_melt,
    x="county",
    y="percentage",
    hue="age_group"
)
plt.title("Counties Where Seniors Outnumber Youth")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Percentage")
plt.tight_layout()
plt.show()


> **Aging Population: Counties Where Seniors Outnumber Youth**  
> In 19 California counties, the percentage of residents aged 65 and older exceeds the share of those aged 0–17.    
> This pattern is most common in rural or retirement-oriented areas, which often experience lower birth rates and youth out-migration.
>
> **Top counties by senior dominance**:
> - **Sierra**: 34.5% seniors vs. 12.8% youth  
> - **Modoc**: 31.0% vs. 19.2%  
> - **Plumas**: 29.8% vs. 17.5%  
> - **Mariposa**: 29.4% vs. 16.8%  
> - **Trinity**: 29.3% vs. 18.3%
>
> These counties may require targeted consideration when evaluating:
> - **Earthquake preparedness for older populations**
> - **Insurance engagement strategies**, as older homeowners may be **more risk-averse**
> - **Aging infrastructure** and the capacity to withstand seismic events
>
> This demographic signal could be especially important when combined with housing age, income, or rural isolation in later modeling.

---
#### ➤ EQ Take-Up Rate in Senior-Dominant Counties
Compare earthquake insurance participation in counties with higher senior populations versus others.


In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(
    data=senior_dominant,
    x="eq_takeup_pct",
    y="county",
    hue="county",
    palette="rocket"
)
plt.title("EQ Insurance Take-Up Rate in Counties with Older Population")
plt.xlabel("Earthquake Insurance Take-Up Rate (%)")
plt.ylabel("County")
plt.grid(axis="x", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()



> **EQ Take-Up in Senior-Dominant Counties**
> - The highest take-up rates among these counties are found in **San Luis Obispo (26.3%)** and **Inyo (23.1%)**, both of which are known for active fault proximity and heightened seismic awareness.
> - Other senior-heavy counties, such as **Sierra**, **Plumas**, and **Tuolumne**, show **take-up rates below 10%**, despite having large elderly populations.  
>
>**Insight**:  
> An older population alone doesn’t guarantee higher insurance participation.  
> Take-up rates in these counties likely reflect a **mix of factors** — including:  
> - Proximity to seismic risk  
> - Insurance affordability
> - Local policy incentives or outreach
>
> This supports the idea that **age needs to be interpreted in context** — especially when used as a predictive signal.


---
#### ➤ Correlation: Senior Minus Youth % vs EQ Take-Up
Test relationship between the magnitude of the senior-youth gap and earthquake insurance take-up rate.


In [ ]:
# Pearson and Spearman correlation
pearson_corr = gdf["senior_minus_youth"].corr(gdf["eq_takeup_pct"], method="pearson")
spearman_corr = gdf["senior_minus_youth"].corr(gdf["eq_takeup_pct"], method="spearman")

print(f"Pearson correlation: {pearson_corr:.2f}")
print(f"Spearman correlation: {spearman_corr:.2f}")

# Visualize
plt.figure(figsize=(7, 5))
sns.regplot(
    data=gdf,
    x="senior_minus_youth",
    y="eq_takeup_pct",
    scatter_kws={"alpha": 0.7},
    line_kws={"color": "darkred"},
)
plt.axhline(y=gdf["eq_takeup_pct"].mean(), linestyle="--", color="gray", label="Avg Take-Up")
plt.axvline(x=0, linestyle="--", color="gray", label="Equal Seniors/Youth")
plt.title("EQ Take-Up Rate vs Senior-Youth % Difference")
plt.xlabel("Senior % - Youth %")
plt.ylabel("Earthquake Insurance Take-Up Rate (%)")
plt.legend()
plt.tight_layout()
plt.show()


> **Correlation: EQ Take-Up vs Senior–Youth % Difference**  
> I created a derived variable `senior_minus_youth`, representing the percentage point difference between the senior (65+) and youth (0–17) populations in each county.
>
> The scatterplot shows a **subtle upward trend**, suggesting that counties with more seniors than youth may exhibit slightly higher earthquake insurance take-up rates.
>
> - **Pearson correlation**: **0.12** → weak linear relationship  
> - **Spearman correlation**: **0.25** → slightly stronger rank-based trend
>
> While the correlations are not strong, they indicate a **potential directional signal**:  
> Counties with aging populations may be **modestly more likely** to invest in earthquake coverage — possibly due to **risk awareness**, **homeownership stability**, or **life-stage factors**.
>
> However, the relationship is far from definitive. Age dynamics should be considered **in combination with other variables** such as income, housing stock, and fault line proximity.

---
#### ➤ Focus on Counties with Senior-Youth Gap > 5%
Highlight counties where seniors outnumber youth by more than 5 percentage points, and examine their EQ insurance behavior.


In [ ]:
# Filter counties where the gap is significant
senior_gap = gdf[gdf["senior_minus_youth"] > 5].copy()
display(senior_gap[["county", "age_65_up_pct", "age_0_17_pct", "senior_minus_youth", "eq_takeup_pct"]])

# Correlation within this subset
pearson_gap, _ = pearsonr(senior_gap["senior_minus_youth"], senior_gap["eq_takeup_pct"])
spearman_gap, _ = spearmanr(senior_gap["senior_minus_youth"], senior_gap["eq_takeup_pct"])

print(f"Pearson correlation (gap > 5%): {pearson_gap:.2f}")
print(f"Spearman correlation (gap > 5%): {spearman_gap:.2f}")


In [ ]:
# Plot: Take-Up vs Senior-Youth Gap, colored by size of gap
plt.figure(figsize=(12, 6))
sns.barplot(
    data=senior_gap.sort_values("eq_takeup_pct", ascending=False),
    x="county",
    y="eq_takeup_pct",
    hue="senior_minus_youth",
    dodge=False,
    palette="coolwarm"
)
plt.title("EQ Take-Up Rate in Counties Where Seniors Outnumber Youth (Gap > 5%)")
plt.ylabel("Earthquake Insurance Take-Up Rate (%)")
plt.xlabel("County")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Senior - Youth % Gap", loc="upper right", bbox_to_anchor=(1.25, 1))
plt.tight_layout()
plt.show()


> **Seniors vs. Youth: Age Gap and EQ Insurance Take-Up**  
> I filtered the dataset to include only counties where the senior population (65+) exceeds the youth population (0–17) by more than 5 percentage points.
> In this subset of 10 counties, I examined whether the size of the senior–youth gap correlates with earthquake insurance behavior.
>
> **Correlation Results (gap > 5%)**:
> - **Pearson**: 0.50 → moderate positive linear relationship
> - **Spearman**: 0.62 → strong monotonic relationship  
>
> These results suggest that in counties with a **significantly older population profile**, earthquake insurance take-up tends to **increase as the age gap widens**.  
> This may reflect:
> - Greater **risk awareness**
> - More **financial stability** among older homeowners
> - An **asset protection mindset** among retirees
>
> However, a closer look reveals an interesting nuance:
> - Counties like **San Luis Obispo** and **Inyo**, which have only a **moderate senior-youth gap (~4%)**, show some of the **highest take-up rates** (26.3% and 23.1%).
> - By contrast, counties with the **widest age gaps** — such as **Sierra**, **Modoc**, and **Trinity** — exhibit **lower take-up rates**, despite having older populations.
>
> **Insight**:  
> While a larger senior population *can* be associated with higher take-up, it is **not the only factor**.  
> In fact, a **balanced age structure** may be more predictive of strong insurance participation than extreme senior dominance.
>
> Further investigation is needed to understand how **age composition interacts with income, housing type, seismic risk**, and **insurance affordability** in driving take-up decisions.


----
#### ✅ Conclusion – Age Composition & Insurance Behavior

- Counties where **seniors outnumber youth** tend to show **slightly higher earthquake insurance take-up rates**, possibly due to greater **risk aversion**, higher **homeownership**, or an **asset protection mindset**
- However, the **correlation is modest**, and several senior-heavy counties remain underinsured
- A **balanced age structure** may be more predictive of higher take-up than extreme age skew
- Age-based signals should be **interpreted in context** — and may gain predictive power when combined with other variables like **housing age**, **income**, or **tenure (owner vs. renter)** in future modeling

---
### 🌎 Group 2: Race & Ethnicity

In this section, I explored the racial and ethnic composition of each county and evaluated how these demographic factors relate to earthquake insurance take-up rates.

I focused on six variables:

- `white_alone_pct`  
- `hispanic_pct`  
- `asian_alone_pct`  
- `black_alone_pct`  
- `american_indian_alone_pct` (analyzed in detail earlier)  
- `native_hawaiian_alone_pct`

---
#### ➤ Review Summary Stats
Descriptive statistics of racial and ethnic group percentages by county.

In [ ]:
race_ethnicity_cols = [
    "white_alone_pct", "black_alone_pct", "american_indian_alone_pct", 
    "asian_alone_pct", "native_hawaiian_alone_pct", "hispanic_pct"
]

gdf[race_ethnicity_cols].describe().T.sort_values("mean", ascending=False)


>- **White Alone %**  
>  Most counties have majority White population (median ~49%). Spread is wide—from 9.4% to 88.8%.  
>- **Hispanic %**  
>  High variation across counties (from 5.9% to 85.6%). Median is 27.1%—shows some counties are majority Hispanic.  
>- **Asian Alone %**  
>  Mean is ~7.8%, but skewed by a few counties (e.g. Santa Clara, SF). Many have low single-digit values.  
>- **Black Alone %**  
>  Generally low percentages (mean ~2.8%), max is 12.6%.  
>- **American Indian Alone %**  
>  Mostly very small values, but a few outliers (Alpine, Inyo, Del Norte). Already analyzed.  
>- **Native Hawaiian / Pacific Islander %**  
>  Very low across the board. Likely not a strong variable unless binned or aggregated.   

---
#### ➤ Barplot: Race/Ethnicity Percentages by County
Visual comparison of White, Hispanic, and Asian population shares across all counties.


In [ ]:
# Select relevant columns
ethnicity_cols = ["county", "white_alone_pct", "hispanic_pct", "asian_alone_pct"]
df_plot = gdf[ethnicity_cols].copy()

# Melt into long format
df_melted = df_plot.melt(id_vars="county", var_name="Ethnicity", value_name="Percentage")

# Plot
plt.figure(figsize=(16, 7))
sns.barplot(
    data=df_melted,
    x="county",
    y="Percentage",
    hue="Ethnicity"
)

plt.title("Race & Ethnicity Percentages by County")
plt.xticks(rotation=90, ha='center')
plt.ylabel("Population Percentage (%)")
plt.xlabel("County")
plt.legend(title="Ethnicity Group")
plt.tight_layout()
plt.show()


> **Quick Observation – Race & Ethnicity by County**  
> The chart highlights significant variation across counties:
> - Some counties are **majority White**, while others are **majority Hispanic** (e.g., Imperial, Tulare)
> - **Asian populations** are concentrated in a few counties (e.g., San Francisco, Santa Clara)
>
> This distribution reflects California’s diverse demographic landscape — which may influence housing, insurance behavior, and risk perception differently across regions.

---
#### ➤ Correlation Race & Ethnicity with `eq_takeup_pct`
Measured Pearson and Spearman correlations between racial composition and earthquake insurance participation.


In [ ]:
for col in [
     "white_alone_pct", "black_alone_pct", "american_indian_alone_pct",
     "asian_alone_pct", "native_hawaiian_alone_pct", "hispanic_pct"
]:
    pearson_corr = gdf[col].corr(gdf["eq_takeup_pct"], method="pearson")
    spearman_corr = gdf[col].corr(gdf["eq_takeup_pct"], method="spearman")
    print(f"📊 {col}")
    print(f"   • Pearson  : {pearson_corr:.2f}")
    print(f"   • Spearman : {spearman_corr:.2f}\n")


> **Correlation: Race & Ethnicity vs EQ Insurance Take-Up**  
> I computed Pearson and Spearman correlations between race/ethnicity variables and earthquake insurance take-up rates.  
> The results show **no strong linear or monotonic relationships** for most groups:
>
> - **White Alone %**: weak positive (Pearson: 0.11)  
> - **Black Alone %**: weak negative (Pearson: -0.13)  
> - **Hispanic %**: weak negative (Pearson: -0.13)  
> - **Asian %**: essentially no correlation (Pearson: -0.01)
> - **Native Hawaiian %**: no meaningful relationship  
> - **American Indian Alone %**: previously explored — shows conflicting signals due to outliers (Pearson: 0.20, Spearman: -0.31)
>
> **Takeaway**:  
> Race and ethnicity variables **alone** do not strongly explain differences in insurance take-up across counties.  
> It's likely that **structural factors** — such as **housing type**, **ownership rates**, **building age**, and **socioeconomic conditions** — play a more direct role, and may interact with race/ethnicity in complex ways.

---
#### ➤ Correlation Race Groups with `owner_pct` and `renter_pct`
Explores how each racial group relates to homeownership vs renting — an important indirect factor in EQ insurance behavior.


In [ ]:
# Define variable groups
race_cols = [
    "white_alone_pct", "black_alone_pct", "asian_alone_pct",
    "hispanic_pct", "native_hawaiian_alone_pct", "american_indian_alone_pct"
]
occupancy_cols = ["owner_pct", "renter_pct"]

# Compute correlations
results = []

for race_col in race_cols:
    for occ_col in occupancy_cols:
        pearson_corr, _ = pearsonr(gdf[race_col], gdf[occ_col])
        spearman_corr, _ = spearmanr(gdf[race_col], gdf[occ_col])
        results.append({
            "Race Group": race_col,
            "Occupancy": occ_col,
            "Pearson": round(pearson_corr, 2),
            "Spearman": round(spearman_corr, 2)
        })

# Convert to DataFrame for display
correlation_df = pd.DataFrame(results)
correlation_df


In [ ]:
# Correlation results
data = {
    "Race Group": [
        "white_alone_pct", "black_alone_pct", "asian_alone_pct", "hispanic_pct",
        "native_hawaiian_alone_pct", "american_indian_alone_pct"
    ],
    "Owner (Pearson)": [0.70, -0.36, -0.56, -0.50, -0.21, 0.34],
    "Renter (Pearson)": [-0.70, 0.36, 0.56, 0.50, 0.21, -0.34]
}

df = pd.DataFrame(data)

# Melt to long format for grouped bar plot
df_melted = df.melt(id_vars="Race Group", 
                    value_vars=["Owner (Pearson)", "Renter (Pearson)"],
                    var_name="Occupancy Type", value_name="Correlation")

# Plot
plt.figure(figsize=(10, 6))
sns.barplot(data=df_melted, x="Race Group", y="Correlation", hue="Occupancy Type")
plt.title("Correlation Between Race/Ethnicity and Housing Occupancy")
plt.xticks(rotation=45, ha="right")
plt.axhline(0, color="gray", linestyle="--")
plt.tight_layout()
plt.show()


> **Race & Ethnicity vs Housing Occupancy**  
> I examined how racial and ethnic composition correlates with **homeownership (owner_pct)** and **rentership (renter_pct)** across counties:
>
> - **White Alone %**: Strong positive correlation with homeownership (Pearson: 0.70)  
> - **Asian %**: Strong negative correlation with homeownership (Pearson: -0.56) — renter-skewed  
> - **Hispanic %**: Also renter-dominant (Pearson: -0.50 for ownership)
> - **Black %**: Moderate renter preference, but not as strong
> - **American Indian %**: The **only minority group** with a positive correlation to homeownership (Pearson: 0.34)
>
> **Takeaway**:  
> Race and ethnicity are **strongly tied to housing tenure** in California counties.    
> Since **renters tend to have lower EQ insurance take-up** (explored later), this adds a structural context for interpreting racial disparities in coverage.    
> Targeted strategies may be needed to address **renter-heavy communities**, particularly where cultural or economic factors intersect.  

---
#### ✅ Conclusion – Race & Ethnicity & Insurance Behavior  

**Race & Ethnicity appear to influence earthquake insurance behavior *indirectly*** — primarily through differences in homeownership:

 - No race or ethnicity group showed a strong direct correlation with earthquake insurance take-up.
 - However, homeownership rates vary **sharply by racial group**:
     - **White populations** are strongly associated with homeownership.
     - **Asian and Hispanic groups** are more associated with renting.
     - **American Indian populations** show a positive correlation with homeownership — a unique pattern among minorities.

 **Key Insight**: Since renters are less likely to purchase earthquake insurance, these housing tenure patterns may partially explain disparities across racial lines.

Further exploration may consider:
 - Whether racial disparities persist after controlling for income, age of housing, and insurance availability.
 - How outreach and education strategies could be adapted to **renter-heavy** or **minority-majority** communities.

---
### 💰 Group 3: Poverty & Insurance

In this section, I explored how poverty levels, homeowners insurance coverage, and modeled earthquake loss influence earthquake insurance take-up across California counties.

I focused on the following key variables:

- `below_poverty_pct` — share of county population living below the federal poverty line  
- `homeowners_coverage_pct` — percent of housing units covered by standard homeowners insurance  
- `modeled_loss_ratio_2018_pct` — estimated average annual earthquake loss as a percentage of building value  
- `eq_takeup_pct` — observed earthquake insurance participation rate  

---
#### ➤ Review Summary Stats
Provides descriptive statistics for poverty rate, homeowners coverage, modeled loss ratio, and earthquake insurance take-up — highlighting distributions and outliers. 


In [ ]:
gdf[[
    "below_poverty_pct",
    "eq_takeup_pct",
    "modeled_loss_ratio_2018_pct",
    "homeowners_coverage_pct"
]].describe().T


>**Summary Observations**    
>- **Poverty Rate (`below_poverty_pct`)** ranges from 6.5% to 20.3%, with most counties clustering around 12–13%.  
>- **Earthquake Take-Up Rate (`eq_takeup_pct`)** is heavily right-skewed — most counties fall below 15%, but a few outliers (e.g. Mono) push the max above 50%.  
>- **Modeled Loss Ratio (`modeled_loss_ratio_2018_pct`)** is tightly distributed (7%–20%) and may be a strong predictor of insurance need/risk perception.  
>- **Homeowners Insurance Coverage (`homeowners_coverage_pct`)** is generally high (median 66.5%), but varies substantially — some counties fall below 50%, indicating low insurance market penetration.
>
>This group of variables provides an important lens into both **financial capacity** and **perceived risk** — two crucial drivers of earthquake insurance participation.

---
#### ➤ Correlation with EQ Take-Up Rate
Measures how each socioeconomic variable relates to earthquake insurance take-up, using both Pearson and Spearman correlation methods.
                                                                                              

In [ ]:
cols_to_check = [ 
    "below_poverty_pct", 
    "modeled_loss_ratio_2018_pct", 
    "homeowners_coverage_pct" 
] 

for col in cols_to_check: 
    pearson_corr = pearsonr(gdf[col], gdf["eq_takeup_pct"])[0] 
    spearman_corr = spearmanr(gdf[col], gdf["eq_takeup_pct"])[0] 
    print(f"📊 {col}") 
    print(f" • Pearson : {pearson_corr:.2f}") 
    print(f" • Spearman : {spearman_corr:.2f}\n")
    

In [ ]:
# Select relevant columns
cols = [
    "below_poverty_pct",
    "modeled_loss_ratio_2018_pct",
    "homeowners_coverage_pct",
    "eq_takeup_pct"
]

# Compute correlation matrix
corr_matrix = gdf[cols].corr(method='pearson')

# Plot heatmap
plt.figure(figsize=(6, 4))
sns.heatmap(
    corr_matrix,
    annot=True,
    cmap="coolwarm",
    vmin=-1, vmax=1,
    fmt=".2f"
)
plt.title("Pearson Correlation with EQ Take-Up Rate")
plt.tight_layout()
plt.show()


>**Correlation Observations**  
>- **Poverty Rate (`below_poverty_pct`)** shows a **moderate negative correlation** with earthquake take-up:  
>  Pearson: **-0.31**, Spearman: **-0.45**  
>  Counties with higher poverty tend to have **lower insurance participation**, likely due to financial barriers.  
>
>- **Modeled Loss Ratio (`modeled_loss_ratio_2018_pct`)** has a **weak and inconsistent correlation**:    
>  Pearson: **-0.11**, Spearman: **-0.33**  
>  While this reflects risk exposure, it doesn't clearly translate into take-up behavior — possibly due to a **disconnect between modeled risk and public perception**.
>
>- **Homeowners Coverage (`homeowners_coverage_pct`)** shows the **strongest (negative) correlation** in this group:  
>  Pearson: **-0.40**, Spearman: **-0.23**  
>  Surprisingly, counties with **higher general homeowners insurance coverage** often have **lower earthquake insurance participation**.
>
>This section underscores how economic vulnerability and insurance accessibility strongly influence earthquake coverage uptake across California counties.

---
#### ➤ EQ Take-Up vs Homeowners Insurance (scatter + barplot)
Visualizes the relationship between general homeowners coverage and earthquake insurance take-up across counties — identifying patterns, gaps, and standout regions.


In [ ]:
gdf[["county", "homeowners_coverage_pct", "eq_takeup_pct"]].sort_values("homeowners_coverage_pct", ascending=False).head(10) 


In [ ]:
# Scatter Plot: Homeowners Insurance vs Earthquake Take-Up Rate

plt.figure(figsize=(8, 6))

# Scatter plot
ax = sns.scatterplot(
    data=gdf,
    x="homeowners_coverage_pct",
    y="eq_takeup_pct",
    s=70,
    color="mediumseagreen",
    edgecolor="black"
)

# Add regression line (no linestyle)
sns.regplot(
    data=gdf,
    x="homeowners_coverage_pct",
    y="eq_takeup_pct",
    scatter=False,
    ax=ax,
    color="black"
)

# Annotate a few key counties (optional logic)
for i, row in gdf.iterrows():
    if row["eq_takeup_pct"] > 20 or row["homeowners_coverage_pct"] < 50:
        ax.text(
            row["homeowners_coverage_pct"] + 0.3,
            row["eq_takeup_pct"] + 0.3,
            row["county"],
            fontsize=9
        )

# Axis labels and title
ax.set_title("EQ Take-Up Rate vs Homeowners Insurance Coverage")
ax.set_xlabel("Homeowners Coverage (%)")
ax.set_ylabel("EQ Take-Up Rate (%)")

plt.tight_layout()
plt.show()


>**Observation: Homeowners Insurance vs EQ Take-Up Rate**  
>- Contrary to expectations, this scatter plot reveals a moderate negative correlation between homeowners coverage and earthquake insurance take-up (Pearson: -0.40).  
>- Counties with higher general homeowners insurance coverage (e.g., Calaveras, Lassen, Amador) often show low EQ take-up rates, while some counties with lower coverage (e.g., Mono, San Luis Obispo, Inyo) exhibit higher take-up.  
>
>This suggests that earthquake insurance behavior is not simply a subset of general insurance uptake, and may be influenced more by perceived seismic risk, housing type, or targeted policy incentives.


In [ ]:
# Barplot: Earthquake Take-Up Rate vs Homeowners Coverage by County

melted = gdf[["county", "eq_takeup_pct", "homeowners_coverage_pct"]].melt(
    id_vars="county", var_name="variable", value_name="value"
)

plt.figure(figsize=(14, 6))
ax = sns.barplot(data=melted, x="county", y="value", hue="variable", palette="Set2")

ax.set_xticks(range(len(gdf["county"])))
ax.set_xticklabels(gdf["county"], rotation=90)

ax.set_ylabel("Percentage (%)")
ax.set_xlabel("County")
ax.set_title("Earthquake Take-Up Rate vs Homeowners Coverage by County")
plt.legend(title="Variable", loc="upper right")
plt.tight_layout()
plt.show()


> **Observation: EQ Take-Up vs Homeowners Coverage by County**  
>This side-by-side barplot highlights the disparity between general homeowners insurance coverage and earthquake insurance take-up across California counties.
>
>Most counties show substantially higher homeowners insurance coverage compared to EQ take-up rates, reinforcing the idea that earthquake insurance is treated as a separate and less common product.
>
>Counties like **Mono**, **San Luis Obispo**, and **San Diego** stand out with notably higher EQ take-up rates, while others (e.g., **Imperial**, **Glenn**, **Colusa**) exhibit very low EQ participation despite moderate homeowners coverage — signaling potential underinsurance gaps.

---
#### ✅ Conclusion – Poverty & Insurance
- **Counties with higher poverty rates tend to have lower earthquake insurance take-up**. This is more pronounced with Spearman correlation (–0.45), indicating a consistent rank-order trend.
- **Homeowners insurance coverage shows a moderate negative correlation** with EQ take-up (Pearson –0.40), possibly reflecting broader patterns of insurance adoption or access.
- **Modeled loss ratio** has a weak correlation with EQ take-up, suggesting that expected earthquake risk alone does not strongly drive insurance behavior.
- Outlier counties — where **EQ take-up is high despite low general homeowners coverage** — may reflect targeted outreach, local experience with recent seismic events, or cultural attitudes toward risk.


---
### 🏠 Group 4: Housing Tenure & Age
In this section, I explored the relationship between housing tenure (owner- vs renter-occupied units) and the age of housing stock in each California county, and how these factors relate to earthquake insurance take-up rates.
I focused on five variables:
- `owner_pct` – percentage of owner-occupied housing units
- `renter_pct` – percentage of renter-occupied housing units
- `pre_1980_pct` – housing units built before 1980 (before seismic building codes)
- `built_1980_1999_pct` – housing units built between 1980–1999
- `built_2000_up_pct` – housing units built in 2000 or later

---
#### ➤ Review Summary Stats
Descriptive statistics of housing tenure and housing age categories across California counties.


In [ ]:
# Calculate and round housing age percentages
gdf = gdf.assign(
    built_1980_1999_pct = (gdf["built_1980_1999"] / gdf["total_housing_units"] * 100).round(2),
    built_2000_up_pct = (gdf["built_2000_up"] / gdf["total_housing_units"] * 100).round(2)
)

gdf[["county", "pre_1980_pct", "built_1980_1999_pct", "built_2000_up_pct"]].head()


In [ ]:
# Summary statistics
gdf[["owner_pct", "renter_pct", "pre_1980_pct", "built_1980_1999_pct", "built_2000_up_pct"]].describe().T


>**Observation:**  
Most California counties have a homeowner majority, with a median `owner_pct` of ~63%.  
Nearly half of housing stock was built before 1980 (median 49.4%), suggesting vulnerability due to older construction.  
Newer housing (2000 and up) is less common, typically below 25% of units.  
This age distribution may influence insurance behavior and perceived seismic risk.

---
#### ➤ Barplot: Housing Age Distribution by County
Visualizes the share of homes built before 1980, between 1980–1999, and after 2000 in each county.


In [ ]:
# Melt the data for grouped barplot
melted_age = gdf[[
    "county",
    "pre_1980_pct",
    "built_1980_1999_pct",
    "built_2000_up_pct"
]].melt(
    id_vars="county",
    var_name="housing_age_group",
    value_name="percentage"
)

# Plot
plt.figure(figsize=(14, 6))
ax = sns.barplot(
    data=melted_age,
    x="county",
    y="percentage",
    hue="housing_age_group",
    palette="Set3"
)

# Rotate county names
ax.set_xticks(range(len(gdf["county"])))
ax.set_xticklabels(gdf["county"], rotation=90)

# Labels and title
ax.set_ylabel("Percentage of Housing Units")
ax.set_xlabel("County")
ax.set_title("Housing Age Distribution by County")
plt.legend(title="Housing Age Group", loc="upper right")
plt.tight_layout()
plt.show()


>**Observations:**  
>- Many counties still have a **majority of homes built before 1980**, including **San Francisco (77%)**, **Marin (76%)**, and **Los Angeles (71%)**.
>- Counties with a **larger share of newer housing (post-2000)** include **Placer (38%)**, **Yuba (33%)**, **Riverside (34.8%)**, and **Imperial (29.8%)** — often areas with recent development or population growth.
>- Only a few counties like **Calaveras** and **San Benito** show a relatively balanced mix across all three age groups.
>
>This breakdown helps contextualize why older housing may be associated with higher EQ take-up in the next plots and correlation analysis.


In [ ]:
# Top 3 counties with highest % of pre-1980 housing
gdf[["county", "pre_1980_pct", "built_1980_1999_pct", "built_2000_up_pct"]].sort_values("pre_1980_pct", ascending=False).head(3)


In [ ]:
# Top 3 counties with highest % of built_2000_up_pct
gdf[["county", "pre_1980_pct", "built_1980_1999_pct", "built_2000_up_pct"]].sort_values("built_2000_up_pct", ascending=False).head(3)


---
#### ➤ Correlation with EQ Take-Up Rate
Examines how homeownership rates and housing age correlate with earthquake insurance adoption.


In [ ]:
# Correlation with EQ Take-Up Rate
for col in ["owner_pct", "renter_pct", "pre_1980_pct", "built_1980_1999_pct", "built_2000_up_pct"]:
    pearson_corr = pearsonr(gdf[col], gdf["eq_takeup_pct"])[0]
    spearman_corr = spearmanr(gdf[col], gdf["eq_takeup_pct"])[0]
    print(f"{col}")
    print(f"   • Pearson  : {pearson_corr:.2f}")
    print(f"   • Spearman : {spearman_corr:.2f}\n")
    

In [ ]:
# Correlation Heatmap
sns.heatmap(
    gdf[["owner_pct", "renter_pct", "pre_1980_pct", "built_1980_1999_pct", "built_2000_up_pct", "eq_takeup_pct"]].corr(),
    annot=True, cmap="coolwarm", vmin=-1, vmax=1
)
plt.title("Correlation – Housing Tenure & Age vs EQ Take-Up")
plt.tight_layout()
plt.show()


>**Observation:**    
>The age of housing stock shows the strongest correlation with EQ insurance take-up:  
>- **Older homes (pre-1980)** are **positively correlated** with take-up *(Spearman: +0.36)*, suggesting that perceived risk or building code concerns may drive coverage.  
>- **Newer homes (built 2000+)** show a **negative correlation** *(Spearman: –0.47)*, possibly due to greater seismic resilience or lower risk perception.  
>- **Tenure status** *(owner vs renter)* shows little to no direct correlation with take-up in this univariate view, though it may interact with other factors.

---
#### ➤ Scatterplots: Housing Age vs EQ Take-Up Rate 
Side-by-side plots showing relationships between EQ take-up and both older and newer housing stock.


In [ ]:
# Side-by-Side Scatterplots: Pre-1980 vs Post-2000
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# Plot 1: Pre-1980 Housing vs EQ Take-Up
sns.scatterplot(data=gdf, x="pre_1980_pct", y="eq_takeup_pct", s=70, color="indianred", edgecolor="black", ax=axes[0])
sns.regplot(data=gdf, x="pre_1980_pct", y="eq_takeup_pct", scatter=False, color="black", ax=axes[0])
axes[0].set_title("EQ Take-Up vs Pre-1980 Housing")
axes[0].set_xlabel("Pre-1980 Housing (%)")
axes[0].set_ylabel("EQ Take-Up Rate (%)")

# # Plot 2: Built 2000+ Housing vsEQ Take-Up
sns.scatterplot(data=gdf, x="built_2000_up_pct", y="eq_takeup_pct", s=70, color="orange", edgecolor="black", ax=axes[1])
sns.regplot(data=gdf,x="built_2000_up_pct", y="eq_takeup_pct", scatter=False, color="black", ax=axes[1])
axes[1].set_title("EQ Take-Up Rate vs Post-2000 Housing (%)")
axes[1].set_xlabel("Built 2000 or Later (%)")
axes[1].set_ylabel("EQ Take-Up Rate (%)")

plt.tight_layout()
plt.show()


>**Observation**  
>These scatterplots reveal contrasting relationships between housing age and earthquake insurance behavior:
>- **Pre-1980 Housing** shows a **positive trend**: counties with more older homes tend to have **higher EQ take-up rates**. This may reflect increased awareness of structural vulnerability or stronger retrofitting and insurance outreach in older housing stock.  
>- **Post-2000 Housing** shows a **moderate negative correlation**: counties with newer housing tend to have **lower EQ take-up**, possibly due to newer homes being built to seismic codes, reducing perceived risk.
>
>This reinforces the idea that **perceived vulnerability** and **housing age** play a role in influencing insurance decisions, independent of income or ownership status.

---
#### ➤ Barplot: Housing Age vs EQ Take-Up Rate by County
County-level comparison of EQ take-up rates alongside share of older (pre-1980) and newer (post-2000) homes.


In [ ]:
# Melt selected variables
melted = gdf[[
    "county",
    "pre_1980_pct",
    "built_2000_up_pct",
    "eq_takeup_pct"
]].melt(
    id_vars="county",
    var_name="variable",
    value_name="value"
)

# Rename for clarity
melted["variable"] = melted["variable"].replace({
    "pre_1980_pct": "Pre-1980 Housing",
    "built_2000_up_pct": "Built 2000+",
    "eq_takeup_pct": "EQ Take-Up Rate"
})

# Plot
plt.figure(figsize=(16, 6))
ax = sns.barplot(data=melted, x="county", y="value", hue="variable", palette="Set2")

# Customize axes
ax.set_title("Pre-1980 vs Post-2000 Housing and EQ Take-Up Rate by County")
ax.set_ylabel("Percentage (%)")
ax.set_xlabel("County")
ax.set_xticks(range(len(gdf["county"])))
ax.set_xticklabels(gdf["county"], rotation=90)

plt.legend(title="Variable", loc="upper right")
plt.tight_layout()
plt.show()


>**Observation – Housing Age vs EQ Take-Up Rate (by County)**  
>- Counties with a **higher share of pre-1980 housing** often show moderate to high EQ take-up rates, supporting the earlier finding that older housing stock may be more likely to carry earthquake insurance.
>- Conversely, **counties with a higher proportion of post-2000 housing** tend to have lower take-up rates, possibly due to newer construction being perceived as lower risk or due to changes in ownership/awareness.
>- Some counties (e.g. **San Luis Obispo, Inyo, Mono**) stand out with high EQ take-up despite varied housing age composition, suggesting that local outreach, seismic activity, or cultural factors may also play a strong role.
>- Overall, housing age patterns align with take-up behavior, reinforcing their value as predictors in modeling.

---
#### ✅ Conclusion – Housing Tenure & Age

- **No significant relationship** between housing tenure (`owner_pct` / `renter_pct`) and earthquake insurance take-up.
- **Older homes (pre-1980)** show a **positive correlation** with EQ take-up (Spearman = +0.36), suggesting that aging housing stock may increase perceived risk and drive demand.
- **Newer homes (built 2000 or later)** show a **negative correlation** with take-up (Spearman = –0.47), likely due to improved building codes and lower perceived vulnerability.
- **Middle-era homes (1980–1999)** have **negligible correlation** with take-up behavior, indicating no clear influence in either direction.


---
### 🏚️ Group 5: Housing Vacancy Types  

In this section, I examined how different types of housing vacancy relate to earthquake insurance take-up across California counties. The goal was to see whether areas with more seasonal, migrant, or other vacant housing show different insurance behavior compared to more permanently occupied housing.  

I focused on the following key variables:  

- `for_rent_pct` — share of housing units currently for rent  
- `rented_not_occupied_pct` — units rented but not yet occupied  
- `for_sale_pct` — share of housing units currently for sale  
- `sold_not_occupied_pct` — units sold but not yet occupied  
- `seasonal_pct` — seasonal or recreational housing units  
- `migrant_pct` — housing units for migrant workers  
- `other_vacant_pct` — other vacant units not classified above  

---
#### ➤ Review Summary Stats
Descriptive statistics of housing vacancy types across California counties, including seasonal units, for-rent, for-sale, migrant, and other vacant categories.


In [ ]:
# Define vacancy-related variables
vacancy_cols = [
    "for_rent_pct",
    "rented_not_occupied_pct",
    "for_sale_pct",
    "sold_not_occupied_pct",
    "seasonal_pct",
    "migrant_pct",
    "other_vacant_pct"
]

gdf[vacancy_cols].describe().T
    

> The summary statistics reveal a wide variation across vacancy types in California counties:
>- **Seasonal housing** shows the highest variability, with a **mean of 40.8%** and values ranging from **4.5% to 92.5%**.
>- **Other vacant units** also make up a large share, averaging **29.2%**, with some counties reaching over **60%**.
>- **For rent** and **for sale** vacancy rates are more moderate on average (**15.1%** and **6.1%**, respectively), but still show high dispersion.
>- **Migrant** and **rented-not-occupied** categories are rare overall, with medians below **3%** and many counties at **0%**.
>
>These differences suggest that vacancy composition varies dramatically across counties, especially for seasonal and other-vacant units.

---
#### ➤ Stacked Bar Chart – Vacancy Distribution by County
Visual breakdown of vacancy type composition by county, showing how seasonal, rental, and other categories contribute to total vacant housing.


In [ ]:
# Sort counties alphabetically (optional)
gdf_sorted = gdf.sort_values("county").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(16, 6))
x = range(len(gdf_sorted))

colors = {
    "for_rent_pct": "#fbb4ae",
    "rented_not_occupied_pct": "#b3cde3",
    "for_sale_pct": "#ccebc5",
    "sold_not_occupied_pct": "#decbe4",
    "seasonal_pct": "#fed9a6",
    "migrant_pct": "#ffffcc",
    "other_vacant_pct": "#e5d8bd"
}

bottom = [0] * len(gdf_sorted)

for col in vacancy_cols:
    values = gdf_sorted[col].values
    ax.bar(x, values, bottom=bottom, label=col.replace("_pct", "").replace("_", " ").title(), color=colors[col])
    bottom = [i + j for i, j in zip(bottom, values)]  

ax.set_xticks(x)
ax.set_xticklabels(gdf_sorted["county"], rotation=90)
ax.set_ylabel("Vacant Units (%)")
ax.set_xlabel("County")
ax.set_title("Stacked Vacancy Type Distribution by County")
ax.legend(title="Vacancy Type", bbox_to_anchor=(1.01, 1), loc="upper left")

plt.tight_layout()
plt.show()


>**Observation**
>- **Seasonal housing** dominates vacancy profiles in many rural counties such as Alpine, Mono, and El Dorado — often exceeding 80% of all vacant units.
>- **Other vacant units** are a major share in counties like Imperial, Sacramento, and Kings — possibly reflecting unoccupied or uninhabitable housing.
>- Urban areas (e.g., San Francisco, Santa Clara) tend to show a more **balanced mix** of vacancy types, including higher shares of rental-related vacancies.
>- This chart provides important context before evaluating how different vacancy types correlate with insurance take-up.


In [ ]:
# View all vacancy percentages by county
gdf[["county"] + vacancy_cols].sample(5)


---
#### ➤ Correlations with EQ Take-Up
Statistical correlations between each vacancy type and earthquake insurance take-up, using Pearson and Spearman coefficients to highlight directional trends.


In [ ]:
for col in vacancy_cols:
    pearson_corr = pearsonr(gdf[col], gdf["eq_takeup_pct"])[0]
    spearman_corr = spearmanr(gdf[col], gdf["eq_takeup_pct"])[0]
    print(f"{col}")
    print(f"   • Pearson  : {pearson_corr:.2f}")
    print(f"   • Spearman : {spearman_corr:.2f}\n")


In [ ]:
# Correlation Heatmap
plt.figure(figsize=(7, 5))
sns.heatmap(
    gdf[vacancy_cols + ["eq_takeup_pct"]].corr(),
    annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".2f"
)
plt.title("Correlation – Vacancy Types vs EQ Take-Up")
plt.tight_layout()
plt.show()


>The correlation analysis between vacancy types and earthquake insurance take-up rates reveals a few modest patterns:  
>- **Other vacant units** show the strongest **negative correlation** (Spearman = –0.39), suggesting that counties with a high share of undefined or miscellaneous vacancies may have lower insurance participation.
>- **Migrant housing** is also negatively correlated (Spearman = –0.39), though it's a rare category overall.
>- **Seasonal housing** shows a **positive correlation** with EQ take-up (Spearman = +0.23), indicating higher participation in areas with many vacation homes.
>- **For sale** and **sold-not-occupied** units show weak negative correlations (Pearson between –0.15 and –0.26).
>- **For rent** and **rented-not-occupied** units exhibit **minimal to no correlation**.
>
>Overall, the relationships are not strong, but suggest that **seasonal use** may increase take-up, while **vacant or transitional units** may suppress it.

---
#### ➤ Scatterplots – Vacancy Type vs EQ Take-Up Rate
Bivariate plots comparing individual vacancy types to EQ insurance take-up, identifying patterns, clusters, and outliers across counties.


In [ ]:
plt.figure(figsize=(14, 6))

# Plot 1: Seasonal Housing %
ax1 = plt.subplot(1, 2, 1)
sns.scatterplot(
    data=gdf, x="seasonal_pct", y="eq_takeup_pct",
    color="teal", s=60, edgecolor="black", ax=ax1
)
sns.regplot(
    data=gdf, x="seasonal_pct", y="eq_takeup_pct",
    scatter=False, color="black", ax=ax1
)
ax1.set_title("EQ Take-Up Rate vs Seasonal Housing (%)")
ax1.set_xlabel("Seasonal Housing (%)")
ax1.set_ylabel("EQ Take-Up Rate (%)")

# Plot 2: Other Vacant Housing %
ax2 = plt.subplot(1, 2, 2)
sns.scatterplot(
    data=gdf, x="other_vacant_pct", y="eq_takeup_pct",
    color="salmon", s=60, edgecolor="black", ax=ax2
)
sns.regplot(
    data=gdf, x="other_vacant_pct", y="eq_takeup_pct",
    scatter=False, color="black", ax=ax2
)
ax2.set_title("EQ Take-Up Rate vs Other Vacant (%)")
ax2.set_xlabel("Other Vacant Units (%)")
ax2.set_ylabel("")

plt.tight_layout()
plt.show()


>**Observation**  
>- Counties with **higher seasonal housing percentages** tend to show **moderate increases** in EQ insurance take-up — possibly reflecting second-home owners with greater financial means or higher risk awareness.
>- In contrast, counties with a **high share of “other vacant” units** (which often include abandoned or unmarketable homes) exhibit a **strong negative correlation** with take-up, indicating **low insurance penetration** in areas with housing instability.
>- These patterns reinforce the importance of housing market conditions in influencing insurance behavior.


---
#### ➤ Choropleth: Seasonal Housing as % of Vacant Units
Geospatial visualization of seasonal housing concentration, highlighting counties with high shares of vacation and second homes.


In [ ]:
gdf_web = gdf.to_crs(epsg=4326)

m = folium.Map(location=[37.5, -119.5], zoom_start=6, tiles="cartodbpositron")

Choropleth(
    geo_data=gdf_web,
    name="Seasonal Vacancy %",
    data=gdf_web,
    columns=["county", "seasonal_pct"],
    key_on="feature.properties.county",
    fill_color="YlGnBu",
    fill_opacity=0.7,
    line_opacity=0.4,
    legend_name="Seasonal Housing (%) by County",
    nan_fill_color="gray",
).add_to(m)

for _, row in gdf_web.iterrows():
    folium.Marker(
        location=[row.geometry.centroid.y, row.geometry.centroid.x],
        icon=None,
        tooltip=f"{row['county']}: {row['seasonal_pct']:.1f}%",
    ).add_to(m)

folium.LayerControl().add_to(m)

m.save("../maps/map_seasonal_vacancy.html")
m


> ⚠️ *Note: Interactive Folium maps do not render in GitHub Notebooks.*  
> 👉 [Click here to view the full map](https://annapiter.github.io/earthquake-insurance-analysis/maps/map_seasonal_vacancy.html)
>
> This choropleth map highlights the **percentage of seasonal housing units** (as a share of all vacant units) across California counties.  
> - **Alpine (92.5%)**, **Mono (86.3%)**, and **Tuolumne (82.8%)** stand out with the **highest seasonal housing rates**, likely reflecting strong ties to tourism and vacation property markets.
> - Coastal and mountain regions generally have higher seasonal vacancy rates, while urban and inland counties show lower values.
> 
> This spatial pattern supports the idea that **seasonality and second homes** may influence local housing dynamics and, potentially, insurance participation.

---
#### ➤ Choropleth: Seasonal Housing % + Earthquake Insurance Take-Up Overlay
Dual-layer map combining seasonal housing rates with EQ insurance participation, offering spatial insight into potential correlations.


In [ ]:
m2 = folium.Map(location=[37.5, -119.5], zoom_start=6, tiles="cartodbpositron")

# Choropleth: Seasonal housing %
Choropleth(
    geo_data=gdf_web,
    name="Seasonal Housing %",
    data=gdf_web,
    columns=["county", "seasonal_pct"],
    key_on="feature.properties.county",
    fill_color="YlGnBu",
    fill_opacity=0.7,
    line_opacity=0.3,
    legend_name="Seasonal Housing (%)",
    nan_fill_color="gray"
).add_to(m2)

# Circle Overlay: EQ Insurance Take-Up Rate
for _, row in gdf_web.iterrows():
    CircleMarker(
        location=[row.geometry.centroid.y, row.geometry.centroid.x],
        radius=row["eq_takeup_pct"] * 0.5,  # adjusted scaling
        color="crimson",
        fill=True,
        fill_opacity=0.6,
        popup=folium.Popup(
            f"<b>{row['county']}</b><br>"
            f"Seasonal Housing: {row['seasonal_pct']:.1f}%<br>"
            f"EQ Take-Up Rate: {row['eq_takeup_pct']:.1f}%",
            max_width=250
        ),
        tooltip=f"{row['county']}: {row['eq_takeup_pct']:.1f}%"
    ).add_to(m2)

# Add Layer Control
folium.LayerControl().add_to(m2)
m2.save("../maps/map_seasonal_plus_eq_takeup.html")
m2


> ⚠️ *Note: Interactive Folium maps do not render in GitHub Notebooks.*  
> 👉 [Click here to view the full map](https://annapiter.github.io/earthquake-insurance-analysis/maps/map_seasonal_plus_eq_takeup.html)
>
> This interactive map overlays **seasonal housing rates** with **earthquake insurance take-up** using choropleth and circle markers.
> - Counties like **Mono** and **Alpine** stand out with **both high seasonal housing** and **above-average take-up rates**.
> - Many rural or mountainous counties (e.g., **Plumas**, **Tuolumne**, **Calaveras**) show **high seasonal use** but **low take-up**, suggesting diverse patterns.
>
> Overall, the relationship between seasonal housing and insurance participation is **non-linear**—some high-seasonal regions insure at high rates, while others don’t. Local risk perception or second-homeowner behavior may be driving this variation.


In [ ]:
# View Top 5 Counties by EQ Take-Up Rate with their Seasonal Housing %
gdf[["county", "seasonal_pct", "eq_takeup_pct"]].sort_values("eq_takeup_pct", ascending=False).head(5)


In [ ]:
# Bottom 5 Counties by EQ Take-Up Rate with Seasonal Housing %
gdf[["county", "seasonal_pct", "eq_takeup_pct"]].sort_values("eq_takeup_pct", ascending=True).head(5)


---
#### ✅ Conclusion – Housing Vacancy Types
- **Seasonal housing share** is **positively correlated** with EQ insurance take-up.  
Several counties with high seasonal housing—like **Mono (86.3%)** and **San Luis Obispo (61.5%)**—also show **above-average take-up** (52.3% and 26.3%, respectively).
- In contrast, counties with **low take-up rates** (e.g., Imperial, Glenn, Colusa) tend to have **moderate or low seasonal housing** and may be driven by other structural or socioeconomic factors.
- **Other vacant units** show the **strongest negative correlation** with insurance take-up (Pearson and Spearman = –0.39), possibly reflecting **neglected or unused units** not actively participating in insurance markets.
- **Migrant housing** is also negatively associated with take-up (Spearman = –0.39), highlighting a disconnect between **temporary housing** and **long-term risk planning**.
- **For-sale** and **sold-not-occupied** units show **mild negative trends**, potentially due to higher turnover or uncertainty around ownership status.

---

### ✅ Summary – Part 2: Thematic Analysis of Demographic & Housing Factors

In this second phase of exploratory analysis, I grouped variables into thematic categories and explored how each influences earthquake insurance take-up (`eq_takeup_pct`) across California counties.  
All variables were normalized as **percentages** to ensure comparability across counties of different sizes.

#### Thematic Groups Analyzed:

- **Age Composition**  
  (% under 18, % over 65)  
  → Older populations show slightly higher take-up rates, but correlations are modest.  
  Balanced age structures may be more predictive than extreme age skews.

- **Race & Ethnicity**  
  (% Hispanic, % Asian, % American Indian, etc.)  
  → No group shows strong direct correlation, but **homeownership patterns** by race explain some disparities:  
  White and American Indian populations are associated with higher homeownership.  
  Hispanic and Asian populations are more likely to rent.    
  Since renters are less likely to purchase earthquake insurance, **tenure differences** may mediate the relationship.  


- **Poverty & Insurance Coverage**  
  (% below poverty line, % homeowners insurance coverage, modeled loss ratio)  
  → Higher poverty correlates with **lower take-up** (Spearman –0.45).  
  General homeowners insurance coverage also shows negative correlation.  
  Modeled earthquake risk does **not** strongly predict take-up, suggesting **risk perception ≠ risk behavior**.

- **Housing Tenure & Age**  
  (% owner vs. renter, % pre-1980, % built 2000+)  
  → No strong relationship with tenure.  
  But **older homes** show **positive correlation** with take-up (Spearman +0.36), while **newer homes** show **negative correlation** (Spearman –0.47) — likely due to perceived vulnerability.

- **Vacancy Characteristics**  
  (% seasonal, migrant, other vacant, etc.)  
  → **Seasonal housing share** is **positively correlated** with take-up.  
  Counties with high vacation/second homes (e.g., Mono, San Luis Obispo) show high EQ insurance participation.  
  → **Other vacant** and **migrant housing** show **strong negative correlations**, suggesting these units are disconnected from insurance markets.  
  → For-sale and sold-not-occupied units trend mildly negative.


#### Key Insight:

While no single variable fully explains insurance behavior, several **patterns emerge**:

- **Socioeconomic factors** (like poverty and general insurance access) play a stronger role than seismic risk alone.
- **Housing characteristics** (age, vacancy type) help identify areas with underinsurance risk.
- **Demographic effects** are nuanced and often mediated by housing tenure.

This thematic deep dive provides a robust foundation for **Part 3**, where I will explore **seismic risk variables** — such as fault proximity, intersection with fault lines, and modeled earthquake losses — to further explain variations in insurance behavior across California.


---
## 🧱 Part 3 – Seismic Risk Analysis  

In this part of the analysis, I evaluate seismic hazard exposure using two complementary perspectives:  
1. Proximity to active fault lines  
2. Historical earthquake activity  

Unlike the demographic or housing characteristics explored earlier, these features capture the physical hazard environment — an essential layer for understanding regional vulnerability and insurance behavior.  

---

#### Fault Proximity & Exposure  

I engineered several spatial features to describe how close each county is to California’s major fault systems:  

- `centroid_to_fault_km` – Distance from county centroid to nearest fault  
- `nearest_fault_km` – Minimum boundary distance to any fault  
- `intersects_fault` – Whether a fault physically intersects the county  
- `n_faults_intersecting` – Number of fault segments crossing the county  
- `fault_length_km` – Total length of intersecting faults (in kilometers)  
- `mean_fault_azimuth_deg` – Average directional orientation of intersecting faults  

Together, these variables quantify geographic proximity and structural exposure to fault zones. I’ll explore them through maps, descriptive statistics, and correlations with insurance take-up rates.  

---

#### Earthquake History  

To complement proximity metrics, I integrated USGS Earthquake Catalog data covering all recorded earthquakes in California with magnitude ≥ 4.0 from 1769 through 2025.  

For each county, I computed:  
- `eq_count` – Total number of earthquakes (M ≥ 4.0)  
- `eq_mean_mag` – Mean magnitude of all recorded events  
- `eq_max_mag` – Maximum magnitude recorded in the county  

This long-term seismic record captures both recent and historic hazard exposure, shaping local perceptions of earthquake risk and potentially influencing insurance behavior.  

---
### 🔹 Fault Proximity & Exposure

---
#### ➤ Review Summary Stats
Quick review of fault-related features and their distribution across counties.


In [ ]:
#Preview dataset
gdf[["county", "centroid_to_fault_km", "nearest_fault_km", "intersects_fault", "n_faults_intersecting", 
     "fault_length_km", "mean_fault_azimuth_deg", "eq_takeup_pct"]].sort_values("eq_takeup_pct", ascending=False).head(10)


In [ ]:
# Summary Statistics
gdf[[
    "centroid_to_fault_km", 
    "nearest_fault_km", 
    "intersects_fault", 
    "n_faults_intersecting", 
    "fault_length_km", 
    "mean_fault_azimuth_deg"
]].describe().T.round(2)


>**Observations**
>
>- The median county has 5 intersecting fault segments totaling ~177 km of fault length.  
>  However, the distribution is highly skewed — a few counties have over 1,000 km of fault coverage, while others have none at all.
>- Centroid distance to nearest fault ranges widely:  
>  median: ~12 km, max: nearly 80 km  
>  This confirms that several inland counties are significantly farther from major fault zones.
>- The `nearest_fault_km` feature is extremely left-skewed:  
>  50% of counties have a fault line intersecting them directly (0 km boundary distance).
>- Fault orientations (`mean_fault_azimuth_deg`) cluster around east-west to northwest directions, but vary by region.
>
>These summary statistics highlight California’s diverse fault exposure landscape — a mix of counties with high seismic complexity and others with little to no fault presence.

---
#### ➤ EQ Take-Up vs Fault Length – Bar & Line Plots
Counties with longer total fault length don’t always have higher earthquake insurance coverage. The following dual-axis charts compare take-up and fault length across counties (sorted by fault length and by take-up).


In [ ]:
# Dual-Axis Plot – Sort by Fault Length
df_sort_fault = gdf[["county", "eq_takeup_pct", "fault_length_km"]].sort_values("fault_length_km", ascending=False)
fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()
bars = ax1.bar(df_sort_fault["county"], df_sort_fault["eq_takeup_pct"], color="mediumseagreen", label="EQ Take-Up (%)")
line = ax2.plot(df_sort_fault["county"], df_sort_fault["fault_length_km"], color="crimson", linewidth=2.5, marker="o", label="Fault Length (km)")
ax1.set_xticks(range(len(df_sort_fault)))
ax1.set_xticklabels(df_sort_fault["county"], rotation=90)
ax1.set_ylabel("EQ Take-Up Rate (%)", color="mediumseagreen")
ax2.set_ylabel("Fault Length (km)", color="crimson")
ax1.set_xlabel("County")
plt.title("EQ Take-Up vs Fault Length (Sorted by Fault Length)")
lines_labels = [bars, line[0]]
labels = [l.get_label() for l in lines_labels]
ax1.legend(lines_labels, labels, loc="upper right")
plt.tight_layout()
plt.savefig("../output/plot_dual_take_up_vs_fault_length_1.png", dpi=300)
plt.show()

# Dual-Axis Plot – Sort by EQ Take-Up
df_sort_takeup = gdf[["county", "eq_takeup_pct", "fault_length_km"]].sort_values("eq_takeup_pct", ascending=False)
fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()
bars = ax1.bar(df_sort_takeup["county"], df_sort_takeup["eq_takeup_pct"], color="mediumseagreen", label="EQ Take-Up (%)")
line = ax2.plot(df_sort_takeup["county"], df_sort_takeup["fault_length_km"], color="crimson", linewidth=2.5, marker="o", label="Fault Length (km)")
ax1.set_xticks(range(len(df_sort_takeup)))
ax1.set_xticklabels(df_sort_takeup["county"], rotation=90)
ax1.set_ylabel("EQ Take-Up Rate (%)", color="mediumseagreen")
ax2.set_ylabel("Fault Length (km)", color="crimson")
ax1.set_xlabel("County")
plt.title("EQ Take-Up vs Fault Length (Sorted by Take-Up Rate)")
lines_labels = [bars, line[0]]
labels = [l.get_label() for l in lines_labels]
ax1.legend(lines_labels, labels, loc="upper right")
plt.tight_layout()
plt.savefig("../output/plot_dual_take_up_vs_fault_length_2.png", dpi=300)
plt.show()


>**Observations**  
>The dual-axis charts above compare earthquake insurance take-up rates (bars) to total fault length intersecting each county (line).  
>
>- **Overall, there is no clear linear relationship** between fault length and take-up.  
>  Several counties with extensive fault systems have surprisingly low coverage, while a few with moderate exposure show relatively high take-up.
>   
>- **High-fault counties with low take-up:**  
>    - San Bernardino (3,093 km): only 7.6% take-up  
>    - Los Angeles (1,853 km): 13.5%  
>    - Riverside (1,648 km): 10.4%   
>  These counties likely face affordability, awareness, or demographic barriers to coverage despite high physical exposure.  
>&nbsp; 
>- **Notable high-coverage counties:**  
>    - Mono (824 km): 52.3% — the highest take-up in the state  
>    - San Luis Obispo (1,322 km): 26.3%  
>  These may represent outliers where risk awareness, income, or marketing significantly influence behavior.  
>&nbsp;
>- **Low fault length often aligns with low coverage** — but not always:  
>    - Several counties with minimal fault exposure (e.g., Sierra, Yuba, Glenn) show <10% take-up, consistent with expectations.  
>    - Others (like Alpine) punch above their weight with moderate take-up despite limited fault length.  
>
>**Conclusion:** Fault length alone is not a strong predictor of insurance behavior. This supports the idea that take-up is influenced by a mix of risk perception, demographics, policy accessibility, and possibly past earthquake experience.


In [ ]:
# Top 5 counties with the longest total fault length
gdf[["county", "eq_takeup_pct", "fault_length_km"]].sort_values("fault_length_km", ascending=False).head()


In [ ]:
# Top 5 counties with the highest earthquake insurance take-up rate
gdf[["county", "eq_takeup_pct", "fault_length_km"]].sort_values("eq_takeup_pct", ascending=False).head()


---
#### ➤ Correlation: Seismic Exposure vs Insurance Take-Up
How strongly do seismic proximity variables correlate with earthquake insurance take-up?


In [ ]:
fault_vars = [
    "centroid_to_fault_km",
    "nearest_fault_km",
    "n_faults_intersecting",
    "fault_length_km" 
]

for col in fault_vars:
    pearson_corr = pearsonr(gdf[col], gdf["eq_takeup_pct"])[0]
    spearman_corr = spearmanr(gdf[col], gdf["eq_takeup_pct"])[0]
    print(f" {col}")
    print(f"   • Pearson  : {pearson_corr:.2f}")
    print(f"   • Spearman : {spearman_corr:.2f}\n")


In [ ]:
# Correlation Matrix: Seismic Exposure vs Insurance Take-Up

# Select relevant variables
seismic_insurance_vars = [
    "centroid_to_fault_km",
    "nearest_fault_km",
    "n_faults_intersecting",
    "fault_length_km",
    "eq_takeup_pct"
]

corr_matrix = gdf[seismic_insurance_vars].corr(method="pearson")

plt.figure(figsize=(6.5, 5))
sns.heatmap(
    corr_matrix,
    annot=True,
    cmap="coolwarm",
    vmin=-1, vmax=1,
    annot_kws={"size": 9},
    fmt=".2f",
    square=True
)

plt.title("Correlation Matrix: Seismic Variables + Take-Up")

plt.tight_layout()
plt.show()


>**Observation**    
>- `centroid_to_fault_km` is negatively correlated with take-up (Pearson = –0.34; Spearman = –0.64).  
>  Counties closer to major faults (based on centroid) tend to have higher insurance adoption, with a strong monotonic trend.
>- `n_faults_intersecting` shows the strongest positive correlation with take-up (Pearson = +0.38; Spearman = +0.60).  
>  Exposure to multiple fault segments appears to significantly influence perceived risk and drive higher coverage.
>- `fault_length_km` also has a moderate positive correlation (Pearson = +0.36; Spearman = +0.56), suggesting that counties with greater cumulative fault presence tend to have higher take-up.
>- `nearest_fault_km` shows only a weak correlation (Pearson = –0.10), likely due to its heavily skewed distribution — over half the counties have a value of 0 (direct contact).
>
>These findings confirm that fault-based seismic exposure features are meaningfully associated with insurance behavior, especially when measuring distance via centroid or capturing structural complexity (e.g., number of faults). They also support the inclusion of these variables in predictive modeling for underinsurance risk.

---
#### ➤ Histogram – Centroid to Fault Distance
Visualize the skewed distribution of centroid-to-fault distance across counties. Most counties are close to faults, but a few are far outliers.


In [ ]:
plt.figure(figsize=(6, 4))
sns.histplot(gdf["centroid_to_fault_km"], bins=20, kde=True, color="lightsteelblue")
plt.title("Distribution of Centroid-to-Fault Distance")
plt.xlabel("Distance to Nearest Fault (km)")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


>**Observation**   
>- Most counties are located within 30 km of a major fault.  
>- The distribution is right-skewed, with a few outliers (up to ~80 km).  
>
>This confirms that most of California is in close proximity to seismic risk, and supports the use of this variable in risk modeling.

---
#### ➤ Annotated Scatterplot – Distance to Fault vs Take-Up
Counties with greater distance to faults sometimes show surprising behavior. Annotate the most notable counties here.


In [ ]:
# Scatterplot: Highlight Key Counties
plt.figure(figsize=(8, 6))
sns.scatterplot(data=gdf, x="centroid_to_fault_km", y="eq_takeup_pct", s=70, color="salmon", edgecolor="black")
sns.regplot(data=gdf, x="centroid_to_fault_km", y="eq_takeup_pct", scatter=False, color="black")

highlight = gdf[(gdf["eq_takeup_pct"] > 20) | (gdf["centroid_to_fault_km"] > 50)]

sns.scatterplot(data=highlight, x="centroid_to_fault_km", y="eq_takeup_pct", s=40, color="white", 
                edgecolor="black", linewidth=1.0,marker="o", label="Notable Counties")

for _, row in highlight.iterrows():
    plt.text(
        row["centroid_to_fault_km"] + 0.7,
        row["eq_takeup_pct"] + 0.7,
        row["county"],
        fontsize=9
    )

plt.title("EQ Take-Up vs Distance to Fault (Centroid)")
plt.xlabel("Distance from County Centroid to Fault (km)")
plt.ylabel("EQ Take-Up Rate (%)")
plt.tight_layout()
plt.savefig("../output/plot_scatter_takeup_vs_centroid.png", dpi=300)
plt.show()


> **Observations**
>
> - A clear inverse relationship is observed: counties closer to major faults (by centroid) tend to have higher earthquake insurance take-up rates.
> - The negative slope of the regression line supports this trend, matching the strong Spearman correlation of –0.64.
> - Several counties with greater distances (>50 km) still show moderate-to-high take-up, possibly due to:
>   - Recent seismic activity
>   - Public education campaigns
>   - Regional spillover from neighboring high-risk zones
> - This plot reinforces the importance of including fault proximity in predictive models and risk assessment frameworks.

---
#### ➤ Scatterplots – Seismic Exposure vs EQ Take-Up
Explore relationships between fault complexity (`n_faults_intersecting`, `fault_length_km`) and earthquake insurance take-up.


In [ ]:
# Scatterplots: n_faults_intersecting and fault_length_km
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: n_faults_intersecting
sns.scatterplot(data=gdf, x="n_faults_intersecting", y="eq_takeup_pct", s=60, color="cornflowerblue", edgecolor="black", ax=axes[0])
sns.regplot(data=gdf, x="n_faults_intersecting", y="eq_takeup_pct", scatter=False, color="black", ax=axes[0])
axes[0].set_title("EQ Take-Up vs Number of Faults")
axes[0].set_xlabel("Number of Fault Segments")
axes[0].set_ylabel("EQ Take-Up Rate (%)")

# Right: fault_length_km
sns.scatterplot(data=gdf, x="fault_length_km", y="eq_takeup_pct", s=60, color="lightcoral", edgecolor="black", ax=axes[1])
sns.regplot(data=gdf, x="fault_length_km", y="eq_takeup_pct", scatter=False, color="black", ax=axes[1])
axes[1].set_title("EQ Take-Up vs Fault Length")
axes[1].set_xlabel("Total Fault Length (km)")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()


>**Observations**  
>
>- Counties with more intersecting faults and greater fault length generally show higher take-up rates, but the relationship is not uniform.
>- Mono County stands out with 16 fault segments and 824 km of faults yet an exceptional 52% take-up — far above the state median.  
>- Other high-exposure counties (e.g., Inyo: 26 faults / 1,952 km, Ventura: 29 faults / 747 km) have only 20–23% take-up, showing that hazard exposure alone does not guarantee high coverage.  
>- Conversely, counties with few or no faults (e.g., Mariposa, Colusa, Glenn) consistently show <10% take-up, which matches expectations.  
>
>These scatterplots confirm the moderate positive trends from the correlation analysis: more complex fault exposure tends to coincide with higher insurance adoption — but strong outliers like Mono illustrate that other factors (income, awareness, local culture) also play a major role.


In [ ]:
# Top 5 counties with the highest earthquake insurance take-up rate
gdf[["county", "n_faults_intersecting", "fault_length_km", "eq_takeup_pct"]].sort_values("eq_takeup_pct", ascending=False).head()


In [ ]:
# Top 10 counties by number of intersecting faults
gdf[["county", "n_faults_intersecting", "fault_length_km", "eq_takeup_pct"]].sort_values("n_faults_intersecting", ascending=False).head(10)


In [ ]:
# Bottom 10 counties by number of intersecting faults
gdf[["county", "n_faults_intersecting", "fault_length_km", "eq_takeup_pct"]].sort_values("n_faults_intersecting").head(10)


---
#### ➤ Counties Intersecting Fault Zones (Binary Map)
Visual reference of counties that directly intersect known fault lines (in red), versus those that do not.


In [ ]:
# Binary map: counties that intersect faults
m_intersect = folium.Map(location=[37.5, -119.5], zoom_start=6, tiles="cartodbpositron")

folium.GeoJson(
    gdf_web,
    name="Intersects Fault",
    style_function=lambda feature: {
        "fillColor": "red" if feature["properties"]["intersects_fault"] else "lightgray",
        "color": "black",
        "weight": 0.5,
        "fillOpacity": 0.7,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["county", "intersects_fault"],
        aliases=["County", "Intersects Fault"],
        localize=True
    )
).add_to(m_intersect)

folium.LayerControl().add_to(m_intersect)
m_intersect.save("../maps/map_fault_intersection.html")
m_intersect


> ⚠️ *Note: Interactive Folium maps do not render in GitHub Notebooks.*  
> 👉 [Click here to view the full map](https://annapiter.github.io/earthquake-insurance-analysis/maps/map_fault_intersection.html)
>
>**Observation** 
>- A large portion of California counties directly intersect fault lines (shown in red).  
>- Inland and northern counties are less exposed, matching expectations from proximity metrics.  
>
>This spatial layer helps contextualize the binary `intersects_fault` feature used in later analysis.


---
#### ➤ Earthquake Take-Up Rate by Fault Intersection (Boxplot)
Does physically intersecting a fault zone affect take-up? Group counties by this binary indicator.


In [ ]:
# Reorder categories to show True
gdf["intersects_fault"] = gdf["intersects_fault"].astype("category").cat.reorder_categories([True, False])

# Binary Group Comparison (intersects_fault)
sns.boxplot(
    data=gdf,
    x="intersects_fault",
    y="eq_takeup_pct",
    hue="intersects_fault",
    palette="pastel",
    legend=False
)
plt.title("EQ Take-Up Rate by Fault Intersection (True/False)")
plt.xlabel("Intersects Fault")
plt.ylabel("EQ Take-Up Rate (%)")
plt.tight_layout()
plt.show()


>**Observation**    
>- Counties that intersect with a fault line (`intersects_fault = True`) exhibit notably higher insurance take-up rates compared to those that do not.  
>- This suggests a clear behavioral relationship between direct seismic exposure and the likelihood of purchasing earthquake insurance.  
>- A few counties with `False` values still show moderate take-up — possibly due to proximity to faults, recent earthquakes, or local risk perception.  
>
>Overall, this supports the inclusion of `intersects_fault` in a composite risk or underinsurance index as a simple but effective binary risk flag.


---
#### ➤ Take-Up by Fault Orientation (Boxplot & Barplot)
Are there orientation-based patterns in earthquake insurance coverage? Compare fault direction groups (e.g., NW–SE vs E–W).


In [ ]:
# Boxplot: EQ Take-Up by Fault Orientation
plt.figure(figsize=(8, 6))
sns.boxplot(
    data=gdf,
    x="fault_orientation",
    y="eq_takeup_pct",
    hue="fault_orientation",
    palette="pastel"
)
plt.title("EQ Take-Up Rate by Fault Orientation")
plt.xlabel("Fault Orientation")
plt.ylabel("EQ Take-Up Rate (%)")
plt.tight_layout()
plt.show()


In [ ]:
# Summary Stats: EQ Take-Up by Fault Orientation
gdf.groupby("fault_orientation")["eq_takeup_pct"].describe().round(2)


In [ ]:
# Mean EQ Take-Up (Barplot with Std Dev)
orientation_order = gdf.groupby("fault_orientation")["eq_takeup_pct"].mean().sort_values(ascending=False).index

plt.figure(figsize=(7, 5))
sns.barplot(
    data=gdf,
    x="fault_orientation",
    y="eq_takeup_pct",
    hue="fault_orientation",
    estimator="mean",
    errorbar="sd",
    palette="pastel",
    edgecolor="black", 
    order=orientation_order
)

plt.title("Mean EQ Take-Up Rate by Fault Orientation")
plt.xlabel("Fault Orientation")
plt.ylabel("EQ Take-Up Rate (%)")
plt.tight_layout()
plt.show()


>**Observation**    
>- Fault orientation shows some variation in insurance take-up rates across counties, but the relationship is moderate and partially driven by spatial clustering.  
>
>- Boxplot analysis reveals that counties with East–West (E–W) and Northwest–Southeast (NW–SE) fault orientations tend to have higher and more variable take-up rates compared to other groups.  
>- Barplot analysis with mean and standard deviation confirms this pattern:  
>    - E–W faults → Mean take-up: 11.23%  
>    - NW–SE faults → Mean: 10.62%  
>    - No fault intersecting → Mean: 5.62%  
>    - N–S (only 1 county) → 3.1% (not generalizable)  
>
>These differences may reflect public awareness of major fault systems (e.g., San Andreas trending NW–SE), or regional variation in seismic communication, risk perception, and policy uptake. Overall, while orientation alone is not a strong predictor, it may serve as a contextual spatial feature in combination with other exposure variables.

---
#### ➤ Folium Choropleth – Fault Length by County
Geographic distribution of total fault length across counties. Helps contextualize seismic complexity before identifying underinsured zones.


In [ ]:
# Plot counties shaded by fault length
m_faults = folium.Map(location=[37.5, -119.5], zoom_start=6, tiles="cartodbpositron")

# Create scale
colormap_fault = linear.YlOrRd_09.scale(
    gdf_web["fault_length_km"].min(),
    gdf_web["fault_length_km"].max()
)
colormap_fault.caption = "Total Fault Length (km)"
colormap_fault.add_to(m_faults)

folium.GeoJson(
    gdf_web,
    name="Fault Length Choropleth",
    style_function=lambda feature: {
        "fillColor": colormap_fault(feature["properties"]["fault_length_km"]),
        "color": "black",
        "weight": 0.5,
        "fillOpacity": 0.7,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["county", "fault_length_km"],
        aliases=["County", "Fault Length (km)"],
        localize=True
    )
).add_to(m_faults)

folium.LayerControl().add_to(m_faults)
m_faults.save("../maps/map_fault_length_by_county.html")
m_faults


> ⚠️ *Note: Interactive Folium maps do not render in GitHub Notebooks.*  
> 👉 [Click here to view the full map](https://annapiter.github.io/earthquake-insurance-analysis/maps/map_fault_length_by_county.html)
>
>**Observation**  
>- Fault length varies widely across counties — some exceed 1,500 km, while others have none at all.  
>- High fault presence clusters in Southern California (e.g., San Bernardino, Los Angeles, Kern).  
>
>This map reinforces the importance of accounting for total seismic complexity, not just proximity.

---
#### ➤ Map: Underinsured Counties in Fault Zones
The following Folium map shows counties that intersect fault lines, have above-median fault length, and below-median insurance coverage.


In [ ]:
# Assign underinsured condition
gdf["underinsured_fault_zone"] = (
    (gdf["intersects_fault"] == True) &
    (gdf["fault_length_km"] > gdf["fault_length_km"].median()) &
    (gdf["eq_takeup_pct"] < gdf["eq_takeup_pct"].median())
)
gdf_web["underinsured_fault_zone"] = gdf["underinsured_fault_zone"]

m3 = folium.Map(location=[37.5, -119.5], zoom_start=6, tiles="cartodbpositron")

colormap = linear.OrRd_09.scale(gdf_web["eq_takeup_pct"].min(), gdf_web["eq_takeup_pct"].max())
colormap.caption = "Earthquake Insurance Take-Up Rate (%)"
colormap.add_to(m3)

folium.GeoJson(
    gdf_web,
    name="EQ Take-Up Choropleth",
    style_function=lambda feature: {"fillColor": colormap(feature["properties"]["eq_takeup_pct"]),
        "color": "black", "weight": 0.5, "fillOpacity": 0.7,
    },
    tooltip=folium.GeoJsonTooltip(fields=["county", "eq_takeup_pct"], aliases=["County", "Take-Up %"], localize=True)
).add_to(m3)

for _, row in gdf_web.iterrows():
    if row["underinsured_fault_zone"]:
        CircleMarker(
            location=[row.geometry.centroid.y, row.geometry.centroid.x],
            radius=6, color="red", fill=True, fill_opacity=0.8,
            tooltip=f"{row['county']} (Take-Up: {row['eq_takeup_pct']:.1f}%, Fault Len: {row['fault_length_km']:.0f} km)"
        ).add_to(m3)

caption_html = """
{% macro html(this, kwargs) %}
<div style="position: fixed; bottom: 10px; left: 10px; width: 300px; 
            z-index:9999; font-size:14px;">
  <p><b>🔴 Underinsured Fault Zone Counties</b><br>
     Red circles = counties with high fault exposure and low insurance take-up.</p>
</div>
{% endmacro %}
"""

caption = MacroElement()
caption._template = Template(caption_html)
m3.get_root().add_child(caption)
LayerControl().add_to(m3)
m3.save("../maps/map_underinsured_fault_zone.html")
m3


> ⚠️ *Note: Interactive Folium maps do not render in GitHub Notebooks.*  
> 👉 [Click here to view the full map](https://annapiter.github.io/earthquake-insurance-analysis/maps/map_underinsured_fault_zone.html)

---
#### ➤ Dual-Axis Plot: Underinsured + Fault Length
Counties most at risk (long fault zones + low coverage) are shown again below for detailed inspection.


In [ ]:
underinsured_df = gdf[gdf["underinsured_fault_zone"]].copy()
underinsured_df = underinsured_df.sort_values("fault_length_km", ascending=False)

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

bars = ax1.bar(underinsured_df["county"], underinsured_df["eq_takeup_pct"], color="mediumseagreen", label="EQ Take-Up Rate (%)")
line = ax2.plot(underinsured_df["county"], underinsured_df["fault_length_km"], 
                color="crimson", linewidth=2.5, marker="o",label="Fault Length (km)")

ax1.set_xticks(range(len(underinsured_df["county"])))
ax1.set_xticklabels(underinsured_df["county"], rotation=45, ha="right")

ax1.set_ylabel("EQ Take-Up Rate (%)", color="mediumseagreen")
ax2.set_ylabel("Fault Length (km)", color="crimson")
ax1.set_xlabel("County")
plt.title("Underinsured High-Fault Counties")

lines_labels = [bars, line[0]]
labels = [l.get_label() for l in lines_labels]
ax1.legend(lines_labels, labels, loc="upper right")

plt.tight_layout()
plt.savefig("../output/plot_bar_underinsured_high_fault_counties.png", dpi=300)
plt.show()


>**Observation**  
>The folium map and dual-axis barplot highlight counties that intersect major fault zones, have above-median fault length, yet exhibit below-median earthquake insurance take-up rates.
>
>- Despite significant seismic exposure (e.g., fault lengths exceeding 500 km in several counties), these counties remain underinsured — indicating substantial coverage gaps in high-risk areas.
>- The barplot confirms that counties like San Bernardino, Humboldt, and Mendocino have extensive fault systems but relatively low coverage levels, pointing to possible affordability, awareness, or cultural barriers.
>- These counties may warrant targeted outreach, public education, or even subsidized policy programs to address this disconnect between physical hazard and insurance behavior.
>
>This underinsured segment should be a priority for future modeling, particularly in building a composite underinsurance index or guiding policy interventions in Phase 4.


---
#### ✅ **Conclusion – Fault Proximity & Exposure**

- **Counties with shorter centroid-to-fault distances** tend to have **higher earthquake insurance take-up rates** (Spearman = –0.64), suggesting residents respond to perceived seismic risk when faults are geographically close.

- **Fault complexity also appears to influence coverage behavior**:  
  - Counties intersecting **multiple faults** (`n_faults_intersecting`) and those with **longer cumulative fault length** (`fault_length_km`) show **moderate positive correlations** with take-up (Spearman = +0.60 and +0.56, respectively).  
  - These features may act as intuitive signals of seismic danger for residents, driving higher insurance adoption.  
&nbsp;  
- In contrast, `nearest_fault_km` is **weakly correlated**, likely due to many counties having a 0 km value (direct contact with minor or inactive faults), limiting its predictive usefulness.

- **Fault orientation** groups (e.g., NW–SE, E–W) show mild differences in take-up, with **NW–SE and E–W trending faults** associated with higher average rates.  
  While not a strong standalone predictor, orientation may still provide spatial context or align with awareness of major fault systems (e.g., the San Andreas).

- Finally, the map and dual-axis barplot identified a subset of **underinsured high-fault counties** — regions with significant fault exposure but below-average insurance take-up. These include: **San Bernardino, Imperial, Humboldt, Mendocino, Siskiyou, Modoc, Shasta, Merced**.  
All of these counties intersect major fault lines, have above-median fault length, and yet exhibit below-median take-up rates.

- These counties may benefit from targeted outreach, public education, or incentive programs in future policy design.

Together, these findings confirm that proximity and structural fault exposure are important drivers of earthquake insurance behavior — and should be integrated into any future underinsurance risk scoring or predictive modeling efforts.


---
### 🔹 Earthquake History – Long-Term Seismic Experience

In this section, I explore **historical earthquake activity** as a complementary dimension to structural fault exposure.  
While fault proximity captures *potential* hazard, earthquake history reflects **experienced seismic risk** — which may have a stronger influence on insurance behavior in some communities.

Using the USGS Earthquake Catalog, I computed the following county-level features for all events with magnitude **≥ 4.0** from **1769 through 2025**:

- `eq_count` – Total number of earthquakes recorded in the county  
- `eq_mean_mag` – Average magnitude of all recorded events  
- `eq_max_mag` – Maximum magnitude recorded in the county  

These variables help quantify the **frequency and severity of seismic events** experienced by each region over time.  
In the next steps, I analyze their distribution, spatial patterns, and relationship with earthquake insurance take-up across California.

---
#### ➤ Review Summary Stats
Quick review of earthquake activity metrics and their distribution across California counties, including total events, average magnitude, and maximum observed magnitude.


In [ ]:
#Preview dataset
gdf[["county", "eq_count", "eq_mean_mag", "eq_max_mag", "eq_takeup_pct"]].sort_values("eq_takeup_pct", ascending=False).head()


In [ ]:
# Summary Statistics
gdf[["eq_count", "eq_mean_mag", "eq_max_mag"]].describe().T.round(2)


>**Observation**
>- Earthquake exposure varies widely across counties — from 0 to over 500 recorded events (M ≥ 4.0).  
>- The median county has experienced only 13 earthquakes, while outliers like Mono (374) and Inyo (204) stand out with significantly more seismic activity.
>- Mean magnitude (`eq_mean_mag`) across counties is typically between 4.3 and 4.7, with some high-risk counties reaching up to 6.45.
>- Maximum magnitude (`eq_max_mag`) also shows wide variation, ranging from non-events (0.0) to major quakes (7.9) — with San Luis Obispo, Inyo, and Santa Cruz among those with high historic peaks.
>
>These statistics suggest that both frequency and severity of past seismic events may play a role in shaping local risk perception and insurance adoption.


---
#### ➤ Top 20 Counties by Earthquake Count
Visualize the top 20 counties with the highest number of earthquakes, along with their insurance take-up rates.


In [ ]:
# Filter Top 20 counties by earthquake count
top20 = gdf[gdf["eq_count"] > 0].sort_values("eq_count", ascending=False).head(20)

norm = mcolors.Normalize(vmin=top20["eq_takeup_pct"].min(), vmax=top20["eq_takeup_pct"].max())
cmap = cm.OrRd
colors = cmap(norm(top20["eq_takeup_pct"]))

fig, ax = plt.subplots(figsize=(12, 5))

bars = ax.bar(top20["county"], top20["eq_count"], color=colors, edgecolor="black")

ax.set_title("Top 20 Counties by Earthquake Count (M ≥ 4.0)", fontsize=13)
ax.set_xlabel("County")
ax.set_ylabel("Earthquake Count")
ax.set_xticks(range(len(top20)))
ax.set_xticklabels(top20["county"], rotation=45, ha="right")

sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label("EQ Insurance Take-Up Rate (%)")

plt.tight_layout()
plt.savefig("../output/plot_bar_top20_by_quake_count.png", dpi=300)
plt.show()


>**Observation**  
> - The counties with the highest number of earthquakes (M ≥ 4.0) include San Bernardino (509), Mono (374), Kern (354), and Imperial (317) — reflecting California's most seismically active regions.
> - Mono County stands out with both high seismic activity and the highest take-up rate (52.3%), suggesting strong risk awareness and insurance responsiveness.
> - In contrast, Imperial County, despite having 317 quakes, shows an extremely low take-up rate of just 1.8%, highlighting a major coverage gap in a high-risk area.
> - Several counties (e.g., Inyo, San Diego, San Luis Obispo) show a healthy balance of exposure and take-up, while others (e.g., Fresno, Humboldt, San Benito) lag behind in coverage despite considerable quake counts.  
> 
> These patterns indicate that earthquake frequency alone doesn't guarantee high insurance participation — other factors like income, outreach, or cultural attitudes may play a role.


In [ ]:
# Top 20 most seismically active counties by number of earthquakes (eq_count)
top20[["county", "eq_count", "eq_takeup_pct"]].sort_values("eq_count", ascending=False)


---
#### ➤ Dual Boxplots – EQ Count, Mean, and Max Magnitude
Compare the distribution of earthquake activity variables across all counties using side-by-side boxplots.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot 1: Boxplot of EQ Count
sns.boxplot(data=gdf, y="eq_count", ax=axes[0], color="lightcoral")
axes[0].set_title("Earthquake Count")
axes[0].set_ylabel("EQ Count")

# Plot 2: Boxplot of Mean and Max Magnitude
sns.boxplot(
    data=gdf[["eq_mean_mag", "eq_max_mag"]],
    ax=axes[1],
    palette=["lightblue", "lightgreen"]
)
axes[1].set_title("EQ Mean & Max Magnitude")
axes[1].set_ylabel("Magnitude")

plt.suptitle("Distribution of Earthquake History Variables", fontsize=14, y=1.03)
plt.tight_layout()
plt.show()


>**Observation**
> - Earthquake counts vary widely across California counties, from none to over 500 recorded events (≥ M4.0). Most counties cluster well below the mean (60.0), with a few high-risk outliers like San Bernardino and Mono.
> - Mean and maximum magnitudes are more stable across regions. The average county experiences mean magnitudes between 4.3–4.6, while the max rarely exceeds 7.9.
>
>The plots confirm that quake frequency (eq_count) is far more skewed than magnitude metrics — suggesting event frequency, not just severity, may influence insurance behavior.

---
#### ➤ Correlation: Earthquake History vs Insurance Take-Up
How strongly do earthquake history variables correlate with earthquake insurance take-up?  
Includes Pearson and Spearman coefficients + correlation heatmap.


In [ ]:
quake_vars = ["eq_count", "eq_mean_mag", "eq_max_mag"]

for col in quake_vars:
    pearson_corr = pearsonr(gdf[col], gdf["eq_takeup_pct"])[0]
    spearman_corr = spearmanr(gdf[col], gdf["eq_takeup_pct"])[0]
    print(f" {col}")
    print(f"   • Pearson  : {pearson_corr:.2f}")
    print(f"   • Spearman : {spearman_corr:.2f}\n")


In [ ]:
# Correlation Matrix: Earthquake History vs Insurance Take-Up
# Select relevant variables
quake_insurance_vars = ["eq_count", "eq_mean_mag", "eq_max_mag", "eq_takeup_pct"]

corr_matrix = gdf[quake_insurance_vars].corr(method="pearson")

plt.figure(figsize=(6.5, 5))
sns.heatmap(
    corr_matrix,
    annot=True,
    cmap="coolwarm",
    vmin=-1, vmax=1,
    annot_kws={"size": 9},
    fmt=".2f",
    square=True
)

plt.title("Correlation Matrix: Earthquake History + Take-Up")

plt.tight_layout()
plt.show()


>**Observation** 
>- EQ insurance take-up is moderately correlated with earthquake frequency and severity: `eq_count` has a Pearson = +0.37 and Spearman = +0.45,  
> suggesting counties with more historical earthquakes tend to show higher take-up rates.  
>- Mean magnitude (`eq_mean_mag`) shows weak linear correlation* (Pearson = +0.14), but a stronger rank correlation (Spearman = +0.40), hinting at a nonlinear or threshold-based relationship.  
>- Maximum magnitude (`eq_max_mag`) is the most strongly correlated variable, with Spearman = +0.53 and Pearson = +0.33 — indicating that counties which have experienced larger quakes are more likely to adopt earthquake insurance.  
>- Overall, historical earthquake activity — especially quake count and maximum magnitude — appears to influence public perception and insurance behavior.  
>
> These insights reinforce the value of incorporating earthquake history into a composite underinsurance risk model.

---
#### ➤ Scatterplots – Earthquake History vs EQ Take-Up
Visual exploration of how `eq_count`, `eq_mean_mag`, and `eq_max_mag` relate to earthquake insurance take-up rates.


In [ ]:
# EQ Take-Up vs Earthquake Count
plt.figure(figsize=(8, 5))
sns.scatterplot(data=gdf, x="eq_count", y="eq_takeup_pct", color="steelblue", edgecolor="black", s=70)
sns.regplot(data=gdf, x="eq_count", y="eq_takeup_pct", scatter=False, color="black")
plt.title("EQ Take-Up vs Earthquake Count")
plt.xlabel("Earthquake Count (≥ M4.0)")
plt.ylabel("EQ Take-Up Rate (%)")
plt.tight_layout()
plt.show()

plt.show()


In [ ]:
# EQ Take-Up vs Max and Mean Magnitude (Side-by-Side)
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

quake_vars = ["eq_max_mag", "eq_mean_mag"]
titles = [
    "EQ Take-Up vs Max Magnitude",
    "EQ Take-Up vs Mean Magnitude"
]

for ax, var, title in zip(axes, quake_vars, titles):
    sns.scatterplot(data=gdf, x=var, y="eq_takeup_pct", ax=ax, color="indianred", edgecolor="black", s=70)
    sns.regplot(data=gdf, x=var, y="eq_takeup_pct", ax=ax, scatter=False, color="black")
    ax.set_title(title)
    ax.set_xlabel(var.replace("_", " ").title())
    ax.set_ylabel("EQ Take-Up Rate (%)")

plt.tight_layout()


>**Observation**    
>- **EQ Take-Up increases with earthquake frequency**: Counties with a higher number of historical quakes (e.g., Mono, San Bernardino, Kern) tend to have noticeably higher take-up rates. This supports the idea that **experience with frequent seismic activity drives insurance behavior**.  
>- While **max magnitude** and **mean magnitude** also show **positive correlations**, the relationship is **less consistent**. Some counties with high-magnitude events (e.g., Imperial, Inyo) still show **low insurance adoption**, suggesting that **a single large quake may not be enough** to shift behavior.  
>- Overall, the **strongest trend is with quake count**, aligning with Spearman’s correlation of **0.45**, while the magnitude-based indicators show **moderate but meaningful associations**.  
>
>These results suggest that **cumulative seismic experience** — not just rare catastrophic events — may be a **key driver of perceived risk and insurance decisions** at the county level.  

---
#### ➤ Map – Underinsured Quake Zone Counties
Mapping counties with above-median earthquake activity (count and magnitude), but below-median earthquake insurance adoption.


In [ ]:
# Define underinsured based on quake history
gdf["underinsured_quake_zone"] = (
    (gdf["eq_count"] > gdf["eq_count"].median()) &
    (gdf["eq_max_mag"] > gdf["eq_max_mag"].median()) &
    (gdf["eq_takeup_pct"] < gdf["eq_takeup_pct"].median())
)

# Filter those counties
underinsured_quake = gdf[gdf["underinsured_quake_zone"] == True].copy()

# Check how many
gdf["underinsured_quake_zone"].value_counts()


In [ ]:
# Underinsured Counties with High Earthquake Activity
underinsured_quake[["county", "eq_count", "eq_max_mag", "eq_takeup_pct"]]


In [ ]:
gdf_web["underinsured_quake_zone"] = gdf["underinsured_quake_zone"]

m_quake = folium.Map(location=[37.5, -119.5], zoom_start=6, tiles="cartodbpositron")

colormap = linear.OrRd_09.scale(gdf_web["eq_takeup_pct"].min(), gdf_web["eq_takeup_pct"].max())
colormap.caption = "Earthquake Insurance Take-Up Rate (%)"
colormap.add_to(m_quake)

folium.GeoJson(
    gdf_web, name="EQ Take-Up Choropleth", style_function=lambda feature: {
        "fillColor": colormap(feature["properties"]["eq_takeup_pct"]),
        "color": "black", "weight": 0.5, "fillOpacity": 0.7,},
    tooltip=folium.GeoJsonTooltip(fields=["county", "eq_takeup_pct"], aliases=["County", "Take-Up %"], localize=True)
).add_to(m_quake)

for _, row in gdf_web.iterrows():
    if row["underinsured_quake_zone"]:
        folium.CircleMarker(
            location=[row.geometry.centroid.y, row.geometry.centroid.x], radius=6, color="red", fill=True, fill_opacity=0.8,
            tooltip=f"{row['county']} (Take-Up: {row['eq_takeup_pct']:.1f}%, EQ Count: {int(row['eq_count'])}, Max Mag: {row['eq_max_mag']:.1f})"
        ).add_to(m_quake)

caption_html = """
{% macro html(this, kwargs) %}
<div style="position: fixed; bottom: 10px; left: 10px; width: 320px; z-index:9999; font-size:14px;">
  <p><b>🔴 Underinsured Quake Zone Counties</b><br>
     Red circles = counties with many quakes, high max magnitude, but low insurance take-up.</p>
</div>
{% endmacro %}
"""
caption = MacroElement()
caption._template = Template(caption_html)
m_quake.get_root().add_child(caption)

m_quake.save("../maps/map_underinsured_quake_zone.html")
m_quake


> ⚠️ *Note: Interactive Folium maps do not render in GitHub Notebooks.*  
> 👉 [Click here to view the full map](https://annapiter.github.io/earthquake-insurance-analysis/maps/map_underinsured_quake_zone.html)

---
#### ➤ Dual-Axis Plot – Underinsured Counties by EQ Count & Take-Up
Barplot of EQ insurance take-up with overlaid earthquake count line for underinsured high-quake counties.


In [ ]:
underinsured_quake = gdf[gdf["underinsured_quake_zone"]].copy()
underinsured_quake = underinsured_quake.sort_values("eq_count", ascending=False)

fig, ax1 = plt.subplots(figsize=(9, 5))
ax2 = ax1.twinx()

bars = ax1.bar(underinsured_quake["county"], underinsured_quake["eq_takeup_pct"], color="teal", label="EQ Take-Up Rate (%)")

line = ax2.plot(underinsured_quake["county"], underinsured_quake["eq_count"], color="darkred", 
                linewidth=2.5, marker="o", label="Earthquake Count")

xticks = range(len(underinsured_quake))
ax1.set_xticks(xticks)
ax1.set_xticklabels(underinsured_quake["county"], rotation=45, ha="right")

ax1.set_ylabel("EQ Take-Up Rate (%)", color="teal")
ax2.set_ylabel("Earthquake Count", color="darkred")
ax1.set_xlabel("County")

plt.title("Underinsured Counties with High Earthquake Activity")
lines_labels = [bars, line[0]]
labels = [l.get_label() for l in lines_labels]
ax1.legend(lines_labels, labels, loc="upper right")

plt.tight_layout()
plt.savefig("../output/plot_bar_underinsured_counties_with_high_quake_activity.png", dpi=300)
plt.show()


> **Observation**  
> - This analysis highlights 5 California counties that exhibit high seismic activity (above-median earthquake count or strong historical quakes) yet maintain below-median earthquake insurance take-up rates.
> - Notable examples include:
>   - San Bernardino: 509 quakes (max magnitude 7.5) but only 7.6% take-up.
>   - Imperial: 317 quakes (6.9 max mag) yet just 1.8% insured — one of the lowest take-up rates in the state despite very high risk.
>   - Fresno and Mendocino also show significant quake history with low take-up below 8%.
> - This underinsured cluster likely reflects a gap between hazard and public preparedness, possibly due to affordability, risk perception, or lack of outreach.
>
> These counties are strong candidates for targeted insurance programs, public education campaigns, or further research into barriers to coverage.

---
#### ✅ Conclusion – Earthquake History

- Counties with **more frequent earthquake activity** tend to have **higher earthquake insurance take-up rates**, as shown by a **moderate positive correlation** between `eq_count` and `eq_takeup_pct` *(Pearson = 0.37, Spearman = 0.45)*.  

- Both **average** and **maximum earthquake magnitudes** also show **positive associations** with take-up, especially `eq_max_mag` *(Spearman = 0.53)*, indicating that residents may respond more strongly to **extreme historical events**.  

- **San Bernardino**, **Imperial**, and **Humboldt** counties rank among the most seismically active but remain **underinsured**, pointing to **risk awareness gaps**, **economic barriers**, or **limited insurance outreach** in these regions.  

- Despite strong historical seismic signals, insurance adoption is **far from uniform**, suggesting that **cultural, socioeconomic, and geographic factors** likely play a role in shaping local insurance behavior.  

These findings reinforce the need for **targeted public education and incentives** in counties with high quake exposure but persistently low insurance take-up.  

---
### ✅ Summary – Part 3: Seismic Risk Analysis

This section explored the role of **seismic proximity and earthquake history** in shaping county-level earthquake insurance behavior in California. Key takeaways include:

**Fault Line Exposure**  
  - Counties that **intersect fault lines** or lie **closer to major faults** tend to have **higher insurance take-up rates**, reflecting sensitivity to visible seismic risk.  
  - Features like **fault length** and **number of intersecting segments** show **positive correlations** with take-up, supporting their value in risk modeling.
  - **Fault orientation** (e.g., NW–SE) may influence public awareness and insurance behavior, with counties along major systems like the San Andreas showing above-average coverage.
  - Identified **8 counties** with **high fault exposure** but **below-median insurance take-up**, suggesting **potential gaps in coverage** where public outreach or incentives may be needed.  
   
**Earthquake History**    
  - Historical seismic activity—particularly **quake frequency** and **maximum magnitudes**—correlates moderately with take-up, implying that **recent or extreme events may influence adoption**.  
  - However, several **high-quake counties remain underinsured** (e.g., Imperial, Fresno), indicating that **hazard alone doesn’t guarantee preparedness**.  
    
#### Key Insight:
While seismic variables are clearly linked to insurance behavior, they don’t fully explain it. A combination of geophysical exposure, public perception, income, housing conditions, and policy awareness is likely driving take-up patterns—highlighting the need for a multifactor risk model in future phases.  


## 🧮 Part 4 – Composite Risk Scoring  
### Building My Composite Risk & Underinsurance Score

In this part of the analysis, I’m developing a **composite index** that highlights California counties with:

- High **seismic exposure** (fault lines, earthquake history)  
- High **structural vulnerability** (older homes, lower income, renter populations)  
- Low **earthquake insurance take-up**

This score will help me answer key questions like:
- Which counties are the most at risk and least insured?  
- Where are the biggest gaps between seismic vulnerability and insurance coverage?

My goal is to build an **Underinsurance Risk Score** that can guide future outreach, policy decisions, and modeling in Phase 4.


---
### 🧮 Step 1 – Seismic Risk Score

To begin building the composite index, I created a **Seismic Risk Score** that captures each county’s geophysical exposure to earthquakes. This score combines fault-related data and historical earthquake activity to reflect localized seismic risk.

The following five components were normalized using MinMax scaling and equally weighted:

- **Proximity to fault lines** *(closer = higher risk)* 
- **Fault length intersecting county** *(in km)*  
- **Number of faults intersecting county boundaries**  
- **Historical earthquake count (M ≥ 4.0)**  
- **Maximum observed earthquake magnitude (1769–2025)**

The normalized values were averaged to compute the final **Seismic Risk Score**, scaled between 0 and 1.  
This continuous score allows for fine-grained comparisons between counties and highlights areas with the **highest physical earthquake exposure**.

Rather than grouping counties into bins, I explored the **distribution of scores** and identified **outliers** and **top-ranking counties**, which will help target underinsured high-risk regions in later analysis.


In [ ]:
# Create a working copy
df = gdf.copy()

# Reverse distance: closer = higher risk
df["proximity_to_fault"] = 1 - MinMaxScaler().fit_transform(df[["centroid_to_fault_km"]])

# Normalize fault-related variables
df["fault_length_norm"] = MinMaxScaler().fit_transform(df[["fault_length_km"]])
df["n_faults_norm"] = MinMaxScaler().fit_transform(df[["n_faults_intersecting"]])

# Normalize earthquake history variables
df["eq_count_norm"] = MinMaxScaler().fit_transform(df[["eq_count"]])
df["eq_max_mag_norm"] = MinMaxScaler().fit_transform(df[["eq_max_mag"]])

# Calculate enhanced seismic risk score
df["seismic_risk_score"] = (
    df["proximity_to_fault"] +
    df["fault_length_norm"] +
    df["n_faults_norm"] +
    df["eq_count_norm"] +
    df["eq_max_mag_norm"]
) / 5

# Visualize the distribution
plt.figure(figsize=(6, 4))
sns.histplot(df["seismic_risk_score"], bins=10, kde=True, color="teal")
plt.title("Distribution of Enhanced Seismic Risk Score")
plt.xlabel("Seismic Risk Score (0–1)")
plt.tight_layout()
plt.show()


> **Observation**  
> The distribution of the **Seismic Risk Score** is **right-skewed**, with the majority of counties concentrated between **0.27 and 0.52**.  
>
> - **Average score**: 0.38  
> - **Range**: 0.11 to 0.94  
> - **Median**: 0.36  
>
> Roughly **two-thirds of counties (34 out of 58)** fall within the middle range of **0.27 to 0.52**, while only **5 counties** exceed **0.60** — and just **1 county** scores above **0.85**.  
> 
> This skewed distribution highlights that **severe seismic exposure is rare and highly concentrated**, making it especially important to identify and prioritize these few high-risk counties for insurance and preparedness efforts.


In [ ]:
# Summary stats
df["seismic_risk_score"].describe().round(3)

In [ ]:
# Count the number of counties in each decile (10 equal-width bins) of the seismic risk score distribution
pd.cut(df["seismic_risk_score"], bins=10).value_counts().sort_index()


In [ ]:
# Display top 10 counties by Seismic Risk Score
df_seismic = df[[
    "county", "seismic_risk_score",
    "proximity_to_fault",
    "fault_length_norm", "n_faults_norm",
    "eq_count_norm", "eq_max_mag_norm"
]].sort_values("seismic_risk_score", ascending=False).reset_index(drop=True)

df_seismic = df_seismic.round(3)
df_seismic.head(10)


In [ ]:
# Barplot of top 10 seismic risk counties
top10 = df_seismic.head(10).copy()

plt.figure(figsize=(10, 5))
sns.barplot(
    data=top10,
    x="seismic_risk_score", y="county", hue="county", legend=False,
    palette="Reds_r"
)
plt.title("Top 10 Counties by Seismic Risk Score")
plt.xlabel("Seismic Risk Score")
plt.ylabel("County")
plt.xlim(0, 1)
plt.tight_layout()
plt.show()

# Heatmap of normalized components
plt.figure(figsize=(10, 6))
sns.heatmap(
    top10.set_index("county").drop(columns="seismic_risk_score"),
    cmap="YlOrRd", annot=True, fmt=".2f", linewidths=0.5
)
plt.title("Normalized Seismic Risk Components (Top 10 Counties)")
plt.tight_layout()
plt.show()


> **Observation**  
> - **San Bernardino County** stands out with the **highest seismic risk**, scoring near 1.0 across all components (faults, earthquakes, proximity).  
> - **Los Angeles** ranks second due to high fault density and closeness to active zones, despite lower historical earthquake count.  
> - Counties like **Inyo** and **Mono** appear due to high historical activity, even with lower fault line complexity.  
> - Coastal counties (**Santa Barbara**, **Humboldt**, **San Luis Obispo**) also make the top 10 — reflecting proximity to major offshore or inland faults with high magnitude potential.
>
> This ranking supports targeted insurance expansion and public preparedness campaigns in counties with high physical risk but potentially lower coverage.

---
#### ✅ Step 1 Summary

The **Seismic Risk Score** provides a geospatial baseline for physical earthquake exposure across California counties.  
It captures both **proximity to faults** and **earthquake history**, allowing us to identify regions of **concentrated geophysical risk**.

This index now serves as the **first pillar** of the composite **Underinsurance Risk Score**, which will integrate **social and structural vulnerability** in the next step.


---
### 🧮 Step 2 – Social Vulnerability Score

This step focuses on building a **Social Vulnerability Score** to identify counties with structural, economic, and demographic factors that may reduce their ability to recover from earthquakes.

I developed two versions of the score using normalized indicators:

- **V1**:  
  • Poverty rate  
  • Pre-1980 housing  
  • Renter share  
  • Owner share (reversed)  
  • Migrant %  

- **V2**:  
  • Replaces migrant % with:  
    - Vacancy rate  
    - Age gap (65+ minus <17 population)

The scores are then compared and correlated with **earthquake insurance take-up** to assess their real-world relevance and inform composite risk modeling.

---

#### ➤ Step 2.1 - Social Vulnerability Score (V1)

V1 includes five key indicators of social and structural vulnerability:

- **Poverty rate** (`below_poverty_pct`)  
- **Older housing** (`pre_1980_pct`)  
- **Renter occupancy** (`renter_pct`)  
- **Migrant population share** (`migrant_pct`)  
- **Owner-occupancy rate** (`owner_pct`, reverse-scaled)

All features were normalized using MinMax scaling (0–1) and averaged to compute the final score.

This version helps highlight counties where **socioeconomic weaknesses** may reduce earthquake resilience — especially when combined with high seismic exposure.


In [ ]:
# Copy the dataframe
df = df.copy()

# Normalize positively correlated vulnerability features
df["poverty_norm"] = MinMaxScaler().fit_transform(df[["below_poverty_pct"]])
df["pre1980_norm"] = MinMaxScaler().fit_transform(df[["pre_1980_pct"]])
df["renter_norm"] = MinMaxScaler().fit_transform(df[["renter_pct"]])
df["migrant_norm"] = MinMaxScaler().fit_transform(df[["migrant_pct"]])

# Reverse scale for owner_pct
df["owner_norm_reversed"] = 1 - MinMaxScaler().fit_transform(df[["owner_pct"]])

# Calculate V1 vulnerability score (equal weighting)
df["vulnerability_score_v1"] = (
    df["poverty_norm"] +
    df["pre1980_norm"] +
    df["renter_norm"] +
    df["migrant_norm"] +
    df["owner_norm_reversed"]
) / 5

# View top counties
df_vul_v1 = df[[
    "county", "vulnerability_score_v1",
    "poverty_norm", "pre1980_norm", "renter_norm", "migrant_norm", "owner_norm_reversed"
]].sort_values("vulnerability_score_v1", ascending=False).round(3)

df_vul_v1.head(10)


In [ ]:
# Histogram of V1 scores
plt.figure(figsize=(6, 4))
sns.histplot(df["vulnerability_score_v1"], bins=10, kde=True, color="mediumslateblue")
plt.title("Distribution of Social Vulnerability Score (V1)")
plt.xlabel("Vulnerability Score (0–1)")
plt.tight_layout()
plt.show()


> **Observation**  
> The distribution of **Social Vulnerability Score (V1)** is slightly **right-skewed**, with the majority of California counties concentrated between **0.30 and 0.50**.  
> 
> - **Average score**: 0.39  
> - **Range**: 0.10 to 0.71  
> - **Median**: 0.40  
> 
> Most counties (33 out of 58) fall within the mid-range of **0.30 to 0.50**, while only a handful (4 total) exceed **0.58**, indicating that **extreme social vulnerability is relatively uncommon but present in specific areas**.
>
> This supports the use of this score to flag **localized vulnerabilities** that may coincide with high seismic exposure or low insurance coverage.


In [ ]:
# Summary stats
df["vulnerability_score_v1"].describe().round(3)


In [ ]:
# Count the number of counties in each decile (10 equal-width bins) of the vulnerability score distribution
pd.cut(df["vulnerability_score_v1"], bins=10).value_counts().sort_index()


---

#### ➤ Step 2.2 - Social Vulnerability Score (V2)

In version 2, I refined the vulnerability score by updating two inputs:

- Replaced **migrant share** with:  
  - **Vacant housing share** (`other_vacant_pct`)  
  - **Age gap**: `% age 65+` minus `% age 0–17`

This revision reflects the fact that **vacancy** and **elderly-skewed populations** often correlate with lower resilience and slower post-disaster recovery.

V2 uses the following six variables:

- Poverty  
- Pre-1980 housing  
- Renter share  
- Owner share (reversed)  
- Vacancy  
- Age gap  

All are normalized and equally weighted.


In [ ]:
# Calculate age gap
df["age_gap"] = df["age_65_up_pct"] - df["age_0_17_pct"]

# Normalize all V2 variables
df["vacant_norm"] = MinMaxScaler().fit_transform(df[["other_vacant_pct"]])
df["age_gap_norm"] = MinMaxScaler().fit_transform(df[["age_gap"]])

# Reuse existing normalized columns for consistency
df["vulnerability_score_v2"] = (
    df["poverty_norm"] +
    df["pre1980_norm"] +
    df["renter_norm"] +
    df["owner_norm_reversed"] +
    df["vacant_norm"] +
    df["age_gap_norm"]
) / 6

# Display top V2 vulnerable counties
df_vul_v2 = df[[
    "county", "vulnerability_score_v2",
    "poverty_norm", "pre1980_norm", "renter_norm",
    "owner_norm_reversed", "vacant_norm", "age_gap_norm"
]].sort_values("vulnerability_score_v2", ascending=False).round(3)

df_vul_v2.head(10)


In [ ]:
# Histogram of V2 scores
plt.figure(figsize=(6, 4))
sns.histplot(df["vulnerability_score_v2"], bins=10, kde=True, color="mediumslateblue")
plt.title("Distribution of Social Vulnerability Score (V2)")
plt.xlabel("Vulnerability Score (0–1)")
plt.tight_layout()
plt.show()


> **Observation**  
> The distribution of **Social Vulnerability Score (V2)** is slightly **left-skewed**, with most counties concentrated between **0.40 and 0.55**.  
> 
> - **Average score**: 0.44  
> - **Range**: 0.16 to 0.72  
> - **Median**: 0.45  
> 
> Nearly **three-quarters of counties (41 out of 58)** fall in the moderate range between **0.40 and 0.55**, indicating that **social vulnerability is more broadly distributed** under this revised scoring method.  
> Only **3 counties** exceed a score of **0.66**, and just **1 county** falls below **0.22**, suggesting V2 has **less extreme spread** than V1 but better mid-range separation.
>
>This smoother distribution makes V2 well-suited for use in composite modeling, as it better distinguishes mid-level vulnerability without over-concentrating scores at either end.
  

In [ ]:
# Summary stats
df["vulnerability_score_v2"].describe().round(3)


In [ ]:
# Count the number of counties in each decile (10 equal-width bins) of the vulnerability score distribution
pd.cut(df["vulnerability_score_v2"], bins=10).value_counts().sort_index()


#### ➤ Step 2.3 – Comparing Social Vulnerability Score V1 vs V2

To evaluate how the inclusion of **vacancy rate** and **age gap** affects the Social Vulnerability Score, I compared V1 and V2 across all California counties using four complementary views:

- **Side-by-side maps** showing the Top 10 most vulnerable counties under each version  
- **Comparison table** showing the Top 10 counties with the largest score increase (V2 – V1)  
- **Bar chart** highlighting counties with the highest shift in vulnerability under V2  
- **Scatterplot** showing the correlation between V1 and V2 scores


In [ ]:
# Get top 10 counties by each score version
top10_v1 = df.nlargest(10, "vulnerability_score_v1")
top10_v2 = df.nlargest(10, "vulnerability_score_v2")

fig, axes = plt.subplots(1, 2, figsize=(10, 6))

# --- V1 Map ---
df.plot(ax=axes[0], color="whitesmoke", edgecolor="gray", linewidth=0.5)
top10_v1.plot(ax=axes[0], color="darkred", edgecolor="black", linewidth=1)
for idx, row in top10_v1.iterrows():
    axes[0].annotate(row["county"], xy=(row.geometry.centroid.x, row.geometry.centroid.y),
        ha="center", fontsize=8, color="black")
axes[0].set_title("Top 10 Vulnerable Counties (V1)", fontsize=13)
axes[0].axis("off")

# --- V2 Map ---
df.plot(ax=axes[1], color="whitesmoke", edgecolor="gray", linewidth=0.5)
top10_v2.plot(ax=axes[1], color="darkred", edgecolor="black", linewidth=1)
for idx, row in top10_v2.iterrows():
    axes[1].annotate(row["county"], xy=(row.geometry.centroid.x, row.geometry.centroid.y),
        ha="center", fontsize=8, color="black")
axes[1].set_title("Top 10 Vulnerable Counties (V2)", fontsize=13)
axes[1].axis("off")

legend_patch = mpatches.Patch(color='darkred', label='Top 10 Counties')
fig.legend(handles=[legend_patch], loc="lower center", ncol=1)
plt.suptitle("Comparison of Social Vulnerability Scores – V1 vs V2", fontsize=15)
plt.tight_layout(rect=[0, 0.05, 1, 0.95])
plt.savefig("../output/plot_comparison_social_vulnerability_scores.png", dpi=300)
plt.show()


In [ ]:
# Create comparison table
score_compare = df[["county", "vulnerability_score_v1", "vulnerability_score_v2"]].copy()
score_compare["delta"] = score_compare["vulnerability_score_v2"] - score_compare["vulnerability_score_v1"]

score_compare_sorted = score_compare.sort_values("delta", ascending=False)

# Show top 10 counties with biggest shift
display(score_compare_sorted.head(10))


In [ ]:
# Bar chart of top 15 counties with highest increase (V2 - V1)
top_shift = score_compare_sorted.head(15)

plt.figure(figsize=(10, 6))
plt.barh(top_shift["county"], top_shift["delta"], color="teal")
plt.xlabel("Change in Vulnerability Score (V2 – V1)")
plt.title("Top Counties with Increased Vulnerability Score in V2")
plt.axvline(0, color="gray", linestyle="--")
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# Scatterplot to compare both versions
plt.figure(figsize=(6, 6))
sns.scatterplot(
    data=score_compare,
    x="vulnerability_score_v1", y="vulnerability_score_v2",
    color="slateblue"
)

plt.axline((0, 0), slope=1, color="gray", linestyle="--")

plt.title("Social Vulnerability Score: V1 vs V2")
plt.xlabel("Vulnerability Score V1")
plt.ylabel("Vulnerability Score V2")
plt.tight_layout()
plt.show()


> **Key Observations**  
> - Counties with **high vacancy rates** and **older populations** moved up in the rankings under V2.  
> - Some counties that were mid-ranked in V1 entered the **Top 10** in V2, indicating a shift in vulnerability focus.  
> - The **bar chart** reveals which counties saw the biggest increase in vulnerability under the revised scoring method.  
> - The **scatterplot** shows a positive correlation between the two versions, but with enough variation to justify using V2 moving forward.
>
>This comparison confirms that V2 captures additional social vulnerability dimensions and will be used in the final composite **Underinsurance Risk Score**.


---
#### ➤ Step 2.4 -  Correlation: Vulnerability Score vs. Earthquake Insurance Take-Up

To validate the relevance of the vulnerability indices, I computed Pearson and Spearman correlations between each version and earthquake insurance take-up rates (`eq_takeup_pct`).


In [ ]:
# Compute Pearson and Spearman correlations
correlations = {
    "V1 (vulnerability_score)": {
        "Pearson": df["vulnerability_score_v1"].corr(df["eq_takeup_pct"], method="pearson"),
        "Spearman": df["vulnerability_score_v1"].corr(df["eq_takeup_pct"], method="spearman"),
    },
    "V2 (vulnerability_score_v2)": {
        "Pearson": df["vulnerability_score_v2"].corr(df["eq_takeup_pct"], method="pearson"),
        "Spearman": df["vulnerability_score_v2"].corr(df["eq_takeup_pct"], method="spearman"),
    }
}

pd.DataFrame(correlations).T.round(3)


> **Observation**:
>- Both scores are **negatively correlated** with insurance take-up, suggesting that counties with higher social vulnerability tend to have **lower participation in earthquake insurance**.
>- V1 showed a slightly stronger monotonic (Spearman) correlation, possibly due to the influence of **migrant_pct**, though interpretability is lower.
>- V2 remains a strong candidate due to its improved **policy relevance** and clearer ties to disaster vulnerability (vacancy, age skew).
>
>Despite modest correlation levels, both versions support the idea that **social vulnerability contributes to underinsurance**, and justify inclusion in the final composite score.

---

#### ✅ Step 2 Summary

The Social Vulnerability Score identifies counties with structural, economic, and demographic factors that may reduce their ability to recover from earthquakes.  

It captures conditions such as poverty, older housing, renter share, vacancy, and age imbalance — helping highlight areas where earthquake impact may be amplified by social fragility.

This index now serves as the **second pillar** of the composite **Underinsurance Risk Score**, alongside seismic exposure and insurance coverage, which will be integrated in the next steps.


---
### 🧾 Step 3 – Insurance Gap Score

The third pillar of the composite Underinsurance Risk Score focuses on **insurance behavior and protection gaps**.  
This step identifies counties where **earthquake risk is high but insurance coverage is low**, using variables that reflect both participation and exposure.

Key indicators include:

- **Earthquake insurance take-up rate** (`eq_takeup_pct`)  
- **Homeowners insurance coverage rate** (`homeowners_coverage_pct`)  
- **Modeled loss ratio** (`modeled_loss_ratio_2018_pct`)

All variables will be normalized using MinMax scaling (0–1), reverse-scaled where needed, and averaged to create a final **Insurance Gap Score**, where higher values indicate greater underinsurance risk.

This score will complement the **Seismic Risk Score** and **Social Vulnerability Score** in building the final composite index.

---
#### Recap: Why These Insurance Variables?

In Part 2 (Group 3: Poverty & Insurance), I investigated the relationship between insurance variables and earthquake take-up:

- **Earthquake Insurance Take-Up Rate** showed expected variation by county and will serve as the primary dependent variable throughout this project.
- **Homeowners Insurance Coverage** was **negatively correlated** with EQ take-up (Pearson –0.40), suggesting earthquake insurance behavior may not track general insurance trends.
- **Modeled Loss Ratio** had weak correlation with EQ take-up, but highlights **underlying physical risk** that may not align with coverage decisions.

These findings support the inclusion of all three indicators in the **Insurance Gap Score**, which is designed to quantify coverage shortfalls in high-risk regions.


In [ ]:
# Reverse-scale: low take-up = high gap
df["eq_takeup_gap"] = 1 - MinMaxScaler().fit_transform(df[["eq_takeup_pct"]])

# Reverse-scale: low homeowners coverage = high gap
df["homeowners_gap"] = 1 - MinMaxScaler().fit_transform(df[["homeowners_coverage_pct"]])

# Keep as-is: high modeled loss = high gap
df["modeled_loss_norm"] = MinMaxScaler().fit_transform(df[["modeled_loss_ratio_2018_pct"]])

# Compute final Insurance Gap Score (equal weighting)
df["insurance_gap_score"] = (
    df["eq_takeup_gap"] +
    df["homeowners_gap"] +
    df["modeled_loss_norm"]
) / 3

# Round selected columns for review
df_insurance = df[[
    "county",
    "insurance_gap_score",
    "eq_takeup_pct",
    "eq_takeup_gap",
    "homeowners_coverage_pct",
    "homeowners_gap",
    "modeled_loss_ratio_2018_pct",
    "modeled_loss_norm"
]].round(3).sort_values("insurance_gap_score", ascending=False)

# Display top 10 counties with highest insurance gap
df_insurance.head(10)


> **Observation**  
> The **Insurance Gap Score** highlights counties where earthquake risk is **not matched by adequate insurance coverage**.  
>- **Mariposa, Fresno, Kings, Merced, and Tulare** top the list with very low earthquake insurance take-up (as low as 3–7%) and **maximum modeled loss ratio** (19.9%), indicating **high exposure and poor coverage**.
>- **Imperial County** stands out with the **lowest take-up rate (1.8%)** despite moderate homeowners coverage and significant modeled risk.
>- **San Francisco** appears in the top 10 due to **extremely low homeowners coverage (30.4%)**, even though its take-up rate (11.5%) is not the lowest — a sign of broad insurance gaps beyond earthquake-specific policies.
>- Several Central Valley counties (Fresno, Kings, Madera, Glenn) show **consistent underinsurance** in both homeowners and earthquake-specific metrics, despite high loss expectations.
>
>This score will serve as the **third and final pillar** of the composite **Underinsurance Risk Score**, capturing behavioral and structural gaps in protection.


In [ ]:
# Histogram: Insurance Gap Score distribution
plt.figure(figsize=(6, 4))
sns.histplot(df["insurance_gap_score"], bins=10, kde=True, color="indianred")
plt.title("Distribution of Insurance Gap Score")
plt.xlabel("Insurance Gap Score (0–1)")
plt.ylabel("Number of Counties")
plt.tight_layout()
plt.show()


> **Observation**  
> The distribution of the **Insurance Gap Score** is fairly **centered**, with most counties clustered between **0.45 and 0.58**.  
>
>- **Average score**: 0.51  
>- **Range**: 0.26 to 0.78  
>- **Median**: 0.50  
>- The largest group of counties (26 out of 58) falls between **0.45 and 0.58**, suggesting **moderate underinsurance gaps** across much of California.
>- Only **5 counties** exceed a score of **0.73**, highlighting a small number of **severely underinsured, high-risk counties**.
>- A few counties (5 total) fall below **0.37**, indicating relatively better coverage or lower modeled loss exposure.
>
>This balanced distribution provides strong contrast for identifying **outlier counties** at both ends of the spectrum in the final composite index.
>
---
#### ✅ Step 3 Summary

The Insurance Gap Score identifies counties where earthquake risk is not matched by adequate insurance coverage.  
It combines earthquake take-up rate, homeowners coverage, and modeled loss ratio into a unified metric of underinsurance exposure.

This index completes the third and final pillar of the composite **Underinsurance Risk Score**, and will now be integrated alongside Seismic and Social Vulnerability scores to highlight the most at-risk, least-insured counties in California.


In [ ]:
# Summary statistics
df["insurance_gap_score"].describe().round(3)


In [ ]:
# Bin counts (10 equal-width bins)
pd.cut(df["insurance_gap_score"], bins=10).value_counts().sort_index()


---
### 🧮 Step 4 – Composite Underinsurance Risk Score

This final step combines the three key dimensions of earthquake risk and protection gaps into a single, interpretable index:

1. **Seismic Risk Score** – Physical exposure based on fault proximity and earthquake history  
2. **Social Vulnerability Score (V2)** – Socioeconomic and structural resilience  
3. **Insurance Gap Score** – Coverage shortfalls despite modeled risk

All three scores are scaled between 0 and 1 and carry equal weight in the composite index.  
The resulting **Underinsurance Risk Score** highlights counties with the **greatest mismatch between earthquake risk and insurance coverage**, helping inform policy, outreach, and resilience planning.

---


In [ ]:
# Compute Composite Underinsurance Risk Score
df["underinsurance_risk_score"] = (
    df["seismic_risk_score"] +
    df["vulnerability_score_v2"] +
    df["insurance_gap_score"]
) / 3

# Histogram of Composite Score
plt.figure(figsize=(6, 4))
sns.histplot(df["underinsurance_risk_score"], bins=10, kde=True, color="darkslateblue")
plt.title("Distribution of Underinsurance Risk Score")
plt.xlabel("Composite Score (0–1)")
plt.ylabel("Number of Counties")
plt.tight_layout()
plt.show()


> **Observation**  
>The final **Underinsurance Risk Score** combines seismic risk, social vulnerability, and insurance coverage gaps — scaled from 0 to 1.
>
>- **Mean**: 0.44  
>- **Range**: 0.28 to 0.64  
>- **Most counties** cluster around the middle (0.38–0.50), with a few outliers on the higher-risk end.  
>- Only a handful of counties exceed **0.60**, indicating **compounding risk factors** across all three dimensions.  
>- The histogram is slightly **right-skewed**, suggesting that **extreme underinsurance** is concentrated in a few counties.
>
>This distribution provides a valuable lens for identifying **high-priority regions** for outreach, mitigation, and policy intervention.


In [ ]:
# Summary statistics
df["underinsurance_risk_score"].describe().round(3)


In [ ]:
# Bin counts (10 equal-width bins)
pd.cut(df["underinsurance_risk_score"], bins=10).value_counts().sort_index()


In [ ]:
# Top 6 Highest-Risk Counties
df_composite = df[[
    "county",
    "underinsurance_risk_score",
    "seismic_risk_score",
    "vulnerability_score_v2",
    "insurance_gap_score",
    "eq_takeup_pct"
]].round(3).sort_values("underinsurance_risk_score", ascending=False)

# Display Top 6
df_composite.head(6)


In [ ]:
# Sort by underinsurance risk score
df_sorted = df.sort_values("underinsurance_risk_score", ascending=False)

fig, ax1 = plt.subplots(figsize=(12, 8))

# Barplot: Underinsurance Risk Score (left axis)
bars = ax1.bar(
    df_sorted["county"],
    df_sorted["underinsurance_risk_score"],
    color="mediumseagreen",
    label="Underinsurance Risk Score"
)
ax1.set_ylabel("Underinsurance Risk Score", color="mediumseagreen")
ax1.tick_params(axis="y", labelcolor="mediumseagreen")
ax1.set_xlabel("County")
ax1.set_xticks(range(len(df_sorted)))
ax1.set_xticklabels(df_sorted["county"], rotation=90)

# Lineplot: EQ Take-Up Rate (right axis)
ax2 = ax1.twinx()
ax2.plot(
    range(len(df_sorted)),
    df_sorted["eq_takeup_pct"],
    color="crimson",
    marker="o",
    label="EQ Take-Up Rate (%)"
)
ax2.set_ylabel("Earthquake Insurance Take-Up (%)", color="crimson")
ax2.tick_params(axis="y", labelcolor="crimson")

plt.title("Underinsurance Risk Score vs Earthquake Insurance Take-Up Rate by County")
fig.tight_layout()
plt.savefig("../output/plot_dual_underinsurance_risk_score_vs_takeup.png", dpi=300)
plt.show()


> **Observations**  
>
>- The **Top 6 counties** with the highest **Underinsurance Risk Scores** are:  
**Los Angeles**, **San Bernardino**, **Imperial**, **San Francisco**, **Humboldt**, and **Inyo**.
>
>- These counties represent different underinsurance dynamics:
>     - **Los Angeles** and **San Bernardino** combine **high seismic risk** with **low insurance take-up**, making them especially exposed in a major event.
>     - **Imperial** stands out with the **lowest EQ take-up rate (1.8%)** and **highest insurance gap score**, despite moderate seismic risk.
>     - **San Francisco** has **high social vulnerability** and **insurance gap**, even with only moderate seismic exposure — possibly due to older housing stock and high coverage costs.
>     - **Inyo** is an exception — despite a high overall risk score, it has the **highest EQ take-up (23.1%)** among the top 6, suggesting stronger local awareness or policy adoption.  
>&nbsp; 
>- The **dual-axis plot** confirms:
>     - Most **high-risk counties** still show **low take-up rates**, underscoring the **disconnect between risk and coverage**.
>    - There is **significant variation** in EQ take-up even among top-risk counties, reinforcing the need for **county-specific strategies**.
>
> These findings support targeted interventions in counties where **earthquake risk is high** but **insurance adoption remains low**.

#### ✅ Step 4 Summary

The **Underinsurance Risk Score** provides a unified index that combines physical seismic risk, social vulnerability, and insurance coverage gaps across California counties.

By averaging the three component scores:  
- **Seismic Risk Score**  
- **Social Vulnerability Score (V2)**  
- **Insurance Gap Score**  

I produced a composite metric scaled between 0 and 1.  
Counties with the highest scores — including **Los Angeles**, **San Bernardino**, **Imperial**, and **San Francisco** — reflect regions where earthquake exposure and vulnerability are not matched by adequate insurance uptake.

This final score serves as a key output for identifying **priority regions** for outreach, education, and policy intervention.


---
### Export: Scored Dataset

Final GeoDataFrame saved in two formats:

- `gdf_ca_scored_v4.parquet` — full fidelity for analysis/modeling  
- `gdf_ca_scored_v4.geojson` — web/GIS-friendly format

Saved to: `../output/`
Used in Phase 4: Modeling & Dashboards


In [ ]:
print(df.geometry.name)
print(df.crs)

# List of temporary or intermediate columns to drop
temp_cols = [
    "fault_length_norm", "n_faults_norm", "eq_count_norm", "eq_max_mag_norm",
    "owner_rev", "vulnerability_score_v1", "eq_takeup_gap", "homeowners_gap", "modeled_loss_norm", 
    "poverty_norm", "pre1980_norm", "renter_norm", "migrant_norm", "owner_norm_reversed", "vacant_norm", "age_gap_norm",
    "distance_bin", "native_pct_bin", "senior_minus_youth", "seniors_gt_youth", "proximity_to_fault"
]

df.drop(columns=[col for col in temp_cols if col in df.columns], inplace=True)


In [ ]:
# Round final scores for presentation
df["seismic_risk_score"] = df["seismic_risk_score"].round(3)
df["vulnerability_score"] = df["vulnerability_score_v2"].round(3)
df["insurance_gap_score"] = df["insurance_gap_score"].round(3)
df["underinsurance_risk_score"] = df["underinsurance_risk_score"].round(3)

# Sort columns so final scores are at the end
final_scores = [
    "seismic_risk_score", 
    "vulnerability_score", 
    "insurance_gap_score", 
    "underinsurance_risk_score"
]
df = df[[col for col in df.columns if col not in final_scores] + final_scores].copy()
df.drop(columns=["vulnerability_score_v2"], inplace=True)

# Save as GeoJSON (for mapping, GitHub, Leaflet, etc.)
df.to_file("../output/gdf_ca_scored_v4.geojson", driver="GeoJSON")
print("✅ GeoJSON saved to: ../output/gdf_ca_scored_v4.geojson")

# Save as GeoParquet (for modeling, dashboards, EDA)
df.to_parquet("../output/gdf_ca_scored_v4.parquet", engine="pyarrow", index=False)
print("✅ Parquet saved to: ../output/gdf_ca_scored_v4.parquet")


---

## ✅ Phase 3 Conclusion – Exploratory Analysis & Composite Risk Scoring

In this phase, I explored the interplay between **seismic risk**, **insurance coverage**, and **socio-demographic vulnerability** across California counties.

Key outputs included:
- Normalized **Seismic Risk Score** based on proximity to fault lines and earthquake history.
- Enhanced **Social Vulnerability Score (V2)** using poverty, housing age, renter %, vacancy, and age gap.
- Insurance behavior patterns analyzed through **take-up rates**, **homeowners coverage**, and **modeled loss ratios**, culminating in an **Insurance Gap Score**.
- All three dimensions were integrated into a composite **Underinsurance Risk Score**, identifying the most at-risk, underprotected counties in the state.

This score now serves as the foundation for Phase 4:  
**Predictive Modeling & Dashboard Visualizations**, where I will further analyze correlations, build interpretable models, and develop visual tools for stakeholders.

The enriched dataset has been saved as `gdf_ca_scored_v4.geojson` and `gdf_ca_scored_v4.parquet`.


---

### 🖋️ Author & Attribution

This notebook is part of the **Earthquake Insurance Risk Analysis in California** project.  
Created and maintained by **Anna Piterskaya**  
Data Scientist | Geospatial & Risk Modeling | [GitHub: annapiter](https://github.com/annapiter)

© 2025 Anna Piterskaya. All rights reserved.

📄 This work is licensed under the **Creative Commons Attribution-NonCommercial 4.0 International License**.  
You may share or adapt this work **with attribution** and **non-commercial use only**.  
🔗 [View License Details](https://creativecommons.org/licenses/by-nc/4.0/)
